# 01b — LLM Bootstrap Labelling (Batch API)

The **Batch API** form of `01_llm_bootstrap_labelling.ipynb`: same inputs, same prompt, same
label set, same passes, same output files and schema — notebooks `02` and `03` cannot tell the
two apart. What changes is the transport. `01` issues one synchronous request per tweet per
pass (~1 h of held-open Colab for 2 000 tweets x 2 passes); this notebook writes each pass to
one JSONL file, submits it as one asynchronous batch job at **half the per-token price**, and
collects the results when the job finishes (`TOKENOPT_REF.md` §10, `LLM_BATCH_JOB_SKILL.md`
§ *Loop or Batch API?*).

**Label set (multi-label):** `definition` and `value` (the tweet raises the question of what art
is / what makes it valuable), seven aesthetic frameworks that answer those questions, and
`none`. The criteria the model reads live in `llm_bootstrap_prompt.md`, embedded in the Prompt
Builder cell by `sync_prompt.py` because Colab does not clone this repo at runtime;
`categories.md` explains how the taxonomy is shaped.

**Inputs:** `llm_bootstrap_dataset.pkl` (carved out by `00_hitl_data_preparation.ipynb`,
disjoint from the Base / HITL / Inference partitions — see `partition_ids.pkl`).

**Outputs:** `llm_bootstrap_labels_<RUN_STAMP>.csv` (HITL schema) and
`llm_bootstrap_labels_full_<RUN_STAMP>.pkl` (+ `confidence`, `rationale`), where `RUN_STAMP` is the
time the jobs were **submitted**; a usage record under the same stamp; the seen-ids basket; and,
under `Classifiers_Data/HITL/batch/`, the request files, the downloaded result files and a state
file holding the job names. When `AUDIT_ENABLED` is on, the CSV also carries a stronger model's
second opinion on the `AUDIT_N` most-engaged `none` rows (`audited`, `audit_*`), run as a third
batch job.

**Runtime:** a plain **CPU runtime with standard RAM**. The notebook does almost nothing itself —
it uploads two files and then polls. A batch job's target turnaround is 24 h and is usually far
less, but it is not under our control: if the jobs are still running when `BATCH_WAIT_MAX_S`
elapses, the Collect cell stops with instructions, nothing is lost, and setting
`RESUME_RUN_STAMP` in Configuration collects them in a later session **without resubmitting**
(creation is not idempotent; a resubmit is a second billed job). Jobs expire 48 h after
submission.

**Spend guards:** `MAX_LLM_TWEETS` rows (the line count of each request file is the binding
layer) and `MAX_SESSION_TOKENS` as a pre-flight ceiling on the upper-bound estimate — a batch job
cannot be stopped partway, so both are enforced **before** upload. Run the **Preflight** section
first: one synchronous call per model, then a two-request inline batch job that proves the batch
path end to end.

---
*Sections below are collapsed by default. Click a heading to expand it.*

# 1 · Setup

Environment switch, dependencies, imports, all tunable parameters, the label taxonomy, the
API client, and the per-tweet classification function. Nothing here calls the API.

## Environment and Paths

In [ ]:
%%time
import os
from pathlib import Path
import sys

# --- ENVIRONMENT SWITCH ---
# True  → local machine with Google Drive Desktop mounted
# False → Google Colab cloud
RUNNING_LOCALLY = False

# --- DATASET TYPE ---
# 'AI'  → AItrust_twits_pruned_dict.json    → Partitioned Data/AI Data/
# 'Art' → AItrust_Art_pruned_twit_dict.json → Partitioned Data/Art Data/
DATASET_TYPE = 'AI'

if RUNNING_LOCALLY:
    _REPO_ROOT = str(Path(os.getcwd()).resolve().parents[1])
    if _REPO_ROOT not in sys.path:
        sys.path.insert(0, _REPO_ROOT)
    BASE_PATH = Path('/Volumes/GoogleDrive/My Drive/Colab Projects/AI Public Trust')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/My Drive/Colab Projects/AI Public Trust')

# Pre-compute critical paths
twits_folder          = BASE_PATH / 'Raw Data/Twits/'
test_folder           = BASE_PATH / 'Raw Data/'
datasets_folder       = BASE_PATH / 'Data Sets'
cleanedds_folder      = BASE_PATH / 'Data Sets/Cleaned Data'
networks_folder       = BASE_PATH / 'Data Sets/Networks/'
literature_folder     = BASE_PATH / 'Literature/'
topic_models_folder   = BASE_PATH / 'Models/Topic Modeling/'
classifiers_folder    = BASE_PATH / 'Models/Classifiers/'
classifiers_folder.mkdir(parents=True, exist_ok=True)
partitioned_folder    = cleanedds_folder / 'Partitioned Data' / f'{DATASET_TYPE} Data'
hitl_folder = datasets_folder / 'Classifiers_Data' / 'HITL'

## Dependencies

In [ ]:
%%time
import os
if not RUNNING_LOCALLY:
    print('Running Colab setup...')
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'google-genai'])
else:
    print('Running locally: skipping Colab setup.')

## Imports

In [ ]:
%%time
import hashlib
import json
import re
import time
import textwrap
from pathlib import Path
import numpy as np
import pandas as pd
import tqdm
from google import genai
from google.genai import types

## Configuration

Every tunable knob for this notebook. `SMOKE_TEST` scales the run down; `MAX_LLM_TWEETS` and
`MAX_SESSION_TOKENS` are hard ceilings that bind either way.

In [ ]:
%%time
# ── Run mode ────────────────────────────────────────────────────────────
SMOKE_TEST   = False       # True → label SMOKE_TEST_N tweets only; flip to False for the full run
SMOKE_TEST_N = 500

# True unassigns the Colab runtime in the final cell, freeing the instance once the run
# is done. It also DESTROYS every variable, including out_df — so the Review section
# (section 5) can no longer be re-run interactively afterwards. That is survivable
# because Review executes BEFORE teardown on a Run All, so its report is already printed
# and saved in the notebook output; and the same data is on disk in
# llm_bootstrap_labels_full.pkl, which Review could be re-pointed at.
# Set back to False when iterating on categories.md and re-reading rows by hand.
# (NOTEBOOK_WRITING_SKILL.md §11: gate the disconnect behind a parameter.)
AUTO_DISCONNECT = True

# Absolute ceiling on how many tweets are ever sent to the LLM in a single run.
# Enforced at three independent layers (dataframe truncation in Load Input, an
# assertion before the request files are built, and the LINE COUNT of each request
# file checked before upload), so it binds even when SMOKE_TEST is False. In batch
# mode the uploaded file is the whole commitment — there is no mid-run counter. The prompt in llm_bootstrap_prompt.md is
# still being tuned - raise this deliberately, never as a side effect of flipping
# SMOKE_TEST to False. Note it caps TWEETS, not calls: with N_LABEL_PASSES passes
# the request ceiling is the product (2,000 x 2 = 4,000).
MAX_LLM_TWEETS = 2_000

# Independent labelling passes over the SAME tweets. 1 = single pass (no agreement data).
# 2 gives test-retest reliability: an LLM label is a measurement, and without a repeat
# there is no error bar on it. Disagreements between passes are the highest-value rows
# for human review — they mark where the criteria under-determine the answer.
# COST AND CALLS SCALE LINEARLY: N passes over N_tweets issues N x N_tweets requests.
N_LABEL_PASSES = 2

# Per-pass temperature. None -> TEMPERATURE for every pass.
#   [0.0, 0.0] measures DETERMINISM (agreement should be near-total; disagreement means
#              the model is unstable even at temperature 0).
#   [0.0, 0.7] measures ROBUSTNESS to sampling — a harsher and more informative test of
#              whether the criteria pin the answer down.
#   [0.0, 0.5] the setting in use: pass 1 stays the deterministic operative label, pass 2
#              samples enough to shake loose the rows where the criteria leave real room.
#              Changes what a disagreement MEANS — it is no longer evidence of model
#              instability but of an underdetermined rule, which is the more useful signal
#              while the fence is being tuned. Expect agreement to fall below the near-100%
#              a [0.0, 0.0] run gives (the 2026-10 run at [0.0, 0.3] agreed on 96.3% of
#              2,000 tweets); that drop is the measurement, not a regression.
PASS_TEMPERATURES = [0.0, 0.5]

# Seed for the fixed permutation the run selects from (see Load Input). Changing it
# re-draws WHICH tweets get labelled, so a previously-labelled set is no longer a
# prefix of the new one. Treat it as frozen once the first real run has happened.
SELECTION_SEED = 42

# Hard ceiling on total tokens (input + output) spent in a single run of this notebook.
# BATCH MODE: a PRE-FLIGHT check only. A batch job cannot be stopped at a token count —
# the request files are the commitment — so the Build Request Files cell refuses to
# write files whose upper-bound estimate (every reply filling MAX_OUTPUT_TOKENS, all
# passes) exceeds this number. Nothing stops a submitted job at 77%.
#
# This is a BACKSTOP, not the binding limit, and it has to be sized against the CALL
# ceiling (MAX_LLM_TWEETS x N_LABEL_PASSES = 4,000), not against the tweet count: at
# ~2,200 tokens per call (~1,650 in with the 10-category prompt + up to 512 out) a run
# that actually reaches the cap costs ~8.8M (the 2026-10 run measured 6.48M over 4,000
# calls, i.e. ~1,620 per call). Sized below that, this fires first and silently truncates
# the run — at 2M it would have stopped a 1,000-tweet two-pass run (~2.2M measured)
# partway through pass 2, and the completeness filter in the Run cell would then have
# discarded every tweet pass 2 never reached.
# The row cap is meant to be what binds; this only fires if something unexpected
# inflates per-request cost — a much longer prompt, or a model whose thinking budget is
# not actually off. If it fires, nothing has been uploaded or billed yet.
MAX_SESSION_TOKENS = 9_000_000

# ── LLM ─────────────────────────────────────────────────────────────────
# Gemini model id. Verify against the Available Models cell below, which lists what this
# key can actually reach — model availability shifts and a retired id 404s every call.
# Measured against this project's key on 2026-08-03 with the real prompt:
#   'gemini-3.1-flash-lite' — DEFAULT. Only candidate that ACCEPTS thinking_budget=0, and
#                             parses reliably (1,028 in / 50-65 out per tweet).
#   'gemini-3.5-flash-lite' — works, but REJECTS thinking_budget=0 (400 INVALID_ARGUMENT),
#                             so thinking cost is uncontrolled.
#   'gemini-3.6-flash'      — rejects thinking_budget=0 AND spent 118 thinking tokens, then
#                             truncated the answer at MAX_OUTPUT_TOKENS -> UNPARSEABLE.
#   'gemini-2.5-*'          — RETIRED for new users: 404 on every call, even though
#                             models.list() still advertises them. Do not use.
# See content/how-to/LLM_ERROR_HANDLING_SKILL.md.
MODEL_NAME  = 'gemini-3.1-flash-lite'
TEMPERATURE = 0.0         # deterministic classification; raise only if you want sampling diversity

# Per-pass model. None -> MODEL_NAME for every pass.
#
# Two temperatures of ONE model share a prior, so their agreement partly measures the
# model's self-consistency. Two DIFFERENT models agreeing is the stronger test: it says
# the criteria survive a change of prior, which is what you actually want to know about
# the fence in llm_bootstrap_prompt.md. A disagreement between models points at the
# criteria; a disagreement between temperatures can just be sampling noise.
#
#   None                                            -> MODEL_NAME for every pass
#   ['gemini-3.1-flash-lite', 'gemini-3.5-flash-lite'] -> cross-model agreement
#
# Every DISTINCT model here is probed for thinking-budget support and preflighted with the
# real prompt before the loop starts, because they disagree about what they accept:
# 3.1-flash-lite takes thinking_budget=0, 3.5-flash-lite and 3.6-flash reject it with a
# 400. Without the per-model probe, a config valid for pass 1 fails on every call of pass 2.
PASS_MODELS = None

# Resolved once here so every cell below agrees on which models are in play.
PASS_MODELS_RESOLVED = list(PASS_MODELS or [MODEL_NAME] * N_LABEL_PASSES)
assert len(PASS_MODELS_RESOLVED) >= N_LABEL_PASSES, (
    f'PASS_MODELS has {len(PASS_MODELS_RESOLVED)} entries but N_LABEL_PASSES is {N_LABEL_PASSES}')
PASS_MODELS_RESOLVED = PASS_MODELS_RESOLVED[:N_LABEL_PASSES]
ACTIVE_MODELS = list(dict.fromkeys(PASS_MODELS_RESOLVED))   # distinct, order preserved

# TOKENOPT_REF.md §4: always cap output tokens. The reply is a JSON ARRAY with one
# {category, confidence, rationale} object per selected category (~50-65 tokens each),
# so the cap has to cover several labels, not one: at 128 a tweet with three labels was
# truncated into a PARSE_ERROR row. 512 fits ~7 objects. If rationales still come back
# truncated (they surface as PARSE_ERROR rows), raise this before anything else.
MAX_OUTPUT_TOKENS = 512

# ── Token accounting (TOKENOPT_REF.md §16-17) ───────────────────────────
# Dollars per 1M tokens, keyed by model so switching MODEL_NAME cannot leave a stale rate
# behind. VERIFY against https://ai.google.dev/pricing before trusting a cost figure —
# provider rates change and these go out of date silently. Reporting only; they do not
# gate the run, so a wrong rate misleads but never overspends.
# Verified 2026-10-19 against https://ai.google.dev/gemini-api/docs/pricing (paid tier,
# text). The 3.6/3.8 Flash rates double on 2027-01-01 and 4 Argon's on 2026-12-01; the
# Batch API halves all of them. Earlier placeholders had 3.1-flash-lite at (0.10, 0.40),
# which understated the 2026-10 run's $2.00 as $0.73.
MODEL_RATES = {
    'gemini-3.1-flash-lite': (0.25, 1.50),
    'gemini-3.5-flash-lite': (0.30, 2.50),
    'gemini-3.6-flash':      (0.75, 3.75),
    'gemini-3.8-flash':      (0.75, 3.75),
    'gemini-3.1-pro-preview': (2.00, 12.00),
    'gemini-4-argon':        (2.00, 10.00),
    'gemini-2.5-flash':      (0.30, 2.50),
    'gemini-2.5-flash-lite': (0.10, 0.40),
}
FALLBACK_RATES = (0.30, 2.50)
_unpriced = [_m for _m in ACTIVE_MODELS if _m not in MODEL_RATES]
if _unpriced:
    print(f'No rates recorded for {_unpriced} — cost figures will use the fallback rate '
          f'{FALLBACK_RATES} and be WRONG. Add them to MODEL_RATES.')

# Cost is accounted at the MOST EXPENSIVE rate in use — the highest input rate and the
# highest output rate across ACTIVE_MODELS, taken independently so the bound holds
# whatever the token mix turns out to be. With mixed models the reported figure is an
# UPPER BOUND on spend rather than the actual bill; that is the point, since a cost
# number that can under-report is worse than one that is merely pessimistic. With a
# single model (the default) it is exact.
# Batch pricing is 50% of the interactive rate for the same model (same pricing page).
# Applied to every cost figure in this notebook, the audit's included; set 1.0 to see
# the interactive-equivalent spend.
BATCH_RATE_FACTOR = 0.5
COST_PER_M_INPUT  = BATCH_RATE_FACTOR * max(MODEL_RATES.get(_m, FALLBACK_RATES)[0] for _m in ACTIVE_MODELS)
COST_PER_M_OUTPUT = BATCH_RATE_FACTOR * max(MODEL_RATES.get(_m, FALLBACK_RATES)[1] for _m in ACTIVE_MODELS)
RATES_ARE_WORST_CASE = len(ACTIVE_MODELS) > 1
if RATES_ARE_WORST_CASE:
    print(f'{len(ACTIVE_MODELS)} models in use {ACTIVE_MODELS} — costing everything at the '
          f'dearest rate (${COST_PER_M_INPUT}/M in, ${COST_PER_M_OUTPUT}/M out): '
          f'reported spend is an upper bound.')
COST_ALERT_USD    = 1.00   # pre-flight prints a loud warning above this
CHARS_PER_TOKEN   = 3.5    # English average, used only for the pre-flight estimate

# Thinking budget. 0 disables the internal reasoning chain, which for one-shot
# classification is pure cost — and worse than cost: gemini-3.6-flash spent 118 thinking
# tokens and then TRUNCATED its answer at MAX_OUTPUT_TOKENS, returning unparseable text.
# Not every model accepts 0 (3.5-flash-lite and 3.6-flash reject it with 400
# INVALID_ARGUMENT); the API Key cell probes once and falls back to the model default if
# so, printing a warning. Set to a positive integer to bound rather than disable thinking.
THINKING_BUDGET = 0

# Strict server-side response schema (TOKENOPT_REF.md §5). When True the API enforces the
# label enum and cannot return prose or fenced JSON. Set False to fall back to asking for
# JSON in the prompt and stripping fences client-side — the notebook's original behaviour.
#
# Verified working on gemini-3.1-flash-lite with the real prompt (2026-08-03). The earlier
# 100% failure rate was model retirement and quota exhaustion, not the schema.
USE_RESPONSE_SCHEMA = True

# ── Batch API ───────────────────────────────────────────────────────────
# Every request of a pass goes into ONE JSONL file, uploaded once and run as ONE batch
# job (TOKENOPT_REF.md §10; LLM_BATCH_JOB_SKILL.md § Loop or Batch API?). Half the
# per-token price, its own rate-limit pool, no client-side retries or 429 handling — and
# asynchronous: target turnaround 24 h (usually far less), hard expiry at 48 h.
#
# Set RESUME_RUN_STAMP to COLLECT a run submitted earlier instead of submitting a new
# one. The state file llm_bootstrap_batch_state_<stamp>.json, written the moment each job
# is created, holds the job names; a session that died while polling resumes by name and
# never resubmits (creation is not idempotent — a resubmit is a second billed job). The
# run's outputs then carry the SUBMIT stamp, not the collect time.
RESUME_RUN_STAMP   = None          # e.g. '20261021_1530'; None → submit a new run
BATCH_POLL_S       = 60            # seconds between state checks; a minutes-to-hours job, do not spin
BATCH_WAIT_MAX_S   = 3 * 3600      # stop polling after this; set RESUME_RUN_STAMP and come back

# Inline-mode smoke job in Preflight: BATCH_SMOKE_N real requests per model, proving the
# batch path end to end (request shape, per-request config, result parsing) before the
# file jobs are submitted. The synchronous probe cannot prove this — batch support is per
# model and the catalogue over-reports. Waits up to BATCH_SMOKE_WAIT_S, then CANCELS the
# smoke job and carries on with a warning rather than blocking the run on queue latency.
BATCH_SMOKE        = True
BATCH_SMOKE_N      = 2
BATCH_SMOKE_WAIT_S = 600

# ── Second opinion on `none` (section 6) ───────────────────────────────
# The positive labels have been spot-checked and show almost no false positives. The
# false NEGATIVES have not: `none` is ~80% of the output and nobody has time to read it.
# A stronger model re-reads a FIXED number of `none`-only rows, chosen by engagement
# (likes + retweets, the tweets that shape the discourse) with a seeded random fill when
# fewer than AUDIT_N rows have any engagement at all. Its verdict is written beside the
# operative label (`audited`, `audit_<label>`, `audit_confidence`, `audit_rationale`),
# never in place of it: a disagreement is a row for a human to read, not a relabel.
AUDIT_ENABLED   = True
AUDIT_MODEL     = 'gemini-4-argon'   # 'gemini-3.8-flash' is ~1/3 the price; both read the same prompt
AUDIT_N         = 500                # rows audited; ALSO the hard cap on audit requests, independent of
                                     # MAX_LLM_TWEETS and enforced as the line count of the audit file
AUDIT_SEED      = 42                 # for the random fill; frozen like SELECTION_SEED
# Models newer than 3.1-flash-lite ignore `temperature` (deprecated 2026-07-21, a 400 in
# future generations) and cannot switch thinking OFF — `thinking_level` bottoms out at
# 'low' on 3.8 Flash / 4 Argon ('minimal' on 3.5 Flash-Lite). Thinking tokens are billed
# as output and count against max_output_tokens, so the audit's cap is raised well above
# the main run's 512: a reply truncated mid-thought is a PARSE_ERROR row paid for twice.
AUDIT_THINKING_LEVEL    = 'low'
AUDIT_MAX_OUTPUT_TOKENS = 4096
# Token backstop for the audit alone (AUDIT_N requests x ~1,650 in + up to 4,096 out ≈ 2.9M).
# Pre-flight only, like MAX_SESSION_TOKENS: checked before the audit file is uploaded.
AUDIT_MAX_TOKENS        = 3_000_000
if AUDIT_ENABLED and AUDIT_MODEL not in MODEL_RATES:
    print(f'No rates recorded for AUDIT_MODEL {AUDIT_MODEL!r} — its cost figures will use '
          f'{FALLBACK_RATES} and be WRONG. Add it to MODEL_RATES.')

# ── I/O ─────────────────────────────────────────────────────────────────
INPUT_PATH        = partitioned_folder / 'llm_bootstrap_dataset.pkl'
# Every artifact of one run carries the same stamp (start date and time), so a labels
# file, its full pickle and its usage record can be matched, and a re-run never overwrites
# an earlier run's labels. Notebooks 02 and 03 read the NEWEST stamped CSV.
RUN_STAMP         = RESUME_RUN_STAMP or time.strftime('%Y%m%d_%H%M')
OUTPUT_CSV        = hitl_folder / f'llm_bootstrap_labels_{RUN_STAMP}.csv'
OUTPUT_PKL        = hitl_folder / f'llm_bootstrap_labels_full_{RUN_STAMP}.pkl'
# Request files, downloaded result files and the job-name state file of every run.
BATCH_FOLDER      = hitl_folder / 'batch'
BATCH_FOLDER.mkdir(parents=True, exist_ok=True)
BATCH_STATE_JSON  = BATCH_FOLDER / f'llm_bootstrap_batch_state_{RUN_STAMP}.json'
if RESUME_RUN_STAMP:
    assert BATCH_STATE_JSON.exists(), (
        f'RESUME_RUN_STAMP={RESUME_RUN_STAMP!r} but {BATCH_STATE_JSON} does not exist')
    print(f'RESUMING run {RUN_STAMP}: its jobs will be collected, not resubmitted.')

## Label Set

The live taxonomy is **ten labels, multi-label**: `definition` and `value` (the tweet raises the
question of what art is / what makes it valuable), the seven frameworks `intentionalism`,
`anti_intentionalism`, `cognitivism`, `expressivism`, `hedonism`, `originality` and
`achievement`, and `none` (the residual bucket, used only when nothing else applies).

This cell defines only the *list*, which is what the response schema's `enum` is built from.
The criteria the model actually reads live in `llm_bootstrap_prompt.md` beside this notebook
and are embedded by the Prompt Builder cell below. `categories.md` explains why the taxonomy
is shaped this way; it no longer carries a second copy of the prompt text.

In [ ]:
%%time
# Closed label set. `none` is the residual bucket, not the negation of `originality` — it
# keeps its meaning when further categories are added.
#
# This list drives the enum in the response schema sent to the API. The prompt states the
# label set in its own words, in llm_bootstrap_prompt.md; the Prompt Builder cell asserts
# the two agree, so adding a category here and not there fails loudly instead of producing
# a model that confidently returns a label the schema rejects.
CATEGORIES: list[str] = [
    'definition',
    'value',
    'intentionalism',
    'anti_intentionalism',
    'cognitivism',
    'expressivism',
    'hedonism',
    'originality',
    'achievement',
    'none',
]

assert CATEGORIES, 'CATEGORIES is empty — fill it in before running.'
print(f'{len(CATEGORIES)} categories: {", ".join(CATEGORIES)}')

## API Key and Client

On Colab the key comes from **Colab Secrets** — sidebar → 🔑 → name it `GEMINI_API_KEY` and
toggle **Notebook access** on for this notebook. Locally it comes from the `GEMINI_API_KEY`
environment variable. The key is never written into this file.

In [ ]:
%%time
API_KEY = ''  # leave empty — populated below from Colab secrets / env var

if not RUNNING_LOCALLY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get('GEMINI_API_KEY')
    except Exception as e:
        print(f'Colab userdata lookup failed: {e}')
else:
    API_KEY = os.environ.get('GEMINI_API_KEY', '')

assert API_KEY, 'GEMINI_API_KEY not set. Add it as a Colab secret or env var before running.'
assert MODEL_NAME, 'MODEL_NAME is empty — set it in the Configuration cell.'
assert all(ACTIVE_MODELS), 'PASS_MODELS contains an empty model name.'

client = genai.Client(api_key=API_KEY)

config_kwargs = dict(
    temperature=TEMPERATURE,
    max_output_tokens=MAX_OUTPUT_TOKENS,   # TOKENOPT_REF.md §4
    response_mime_type='application/json',
)

# TOKENOPT_REF.md §5 — native structured output. The schema is enforced server-side, so
# the model cannot return a label outside CATEGORIES and cannot wrap its JSON in prose or
# markdown fences (the two things that produced PARSE_ERROR rows). The enum is derived
# from CATEGORIES, so adding a category needs no edit in this cell.
RESPONSE_SCHEMA = {
    'type': 'array',
    'items': {
        'type': 'object',
        'properties': {
            'category':   {'type': 'string', 'enum': CATEGORIES},
            'confidence': {'type': 'number'},
            'rationale':  {'type': 'string'},
        },
        'required': ['category', 'confidence', 'rationale'],
    }
}
if USE_RESPONSE_SCHEMA:
    if 'response_json_schema' in types.GenerateContentConfig.model_fields:
        config_kwargs['response_json_schema'] = RESPONSE_SCHEMA
    else:
        # older google-genai releases name the field response_schema
        config_kwargs['response_schema'] = RESPONSE_SCHEMA
    print('Strict response schema ENABLED — run the Preflight cell before the loop.')
else:
    print('Strict response schema disabled; falling back to prompt-instructed JSON '
          '+ client-side fence stripping.')
# Thinking budget, probed rather than assumed. Model families disagree about whether
# thinking_budget can be set at all, and guessing wrong costs a whole run: the failure is
# a 400 on every call. One 2-token probe settles it here instead.
def _probe(kwargs, model) -> str:
    """Return '' if the config is accepted by that model, else the error text."""
    try:
        client.models.generate_content(
            model=model, contents='ping',
            config=types.GenerateContentConfig(**{**kwargs, 'max_output_tokens': 8}))
        return ''
    except Exception as e:
        return f'{type(e).__name__}: {str(e)[:110]}'

# Probed PER MODEL, because models disagree about whether thinking_budget can be set at
# all. One shared config would be valid for pass 1's model and 400 on every call of
# pass 2's. MODEL_KWARGS holds the accepted config for each distinct model; the run loop
# layers that pass's temperature on top.
MODEL_KWARGS: dict = {}
for _m in ACTIVE_MODELS:
    _wanted = {**config_kwargs,
               'thinking_config': types.ThinkingConfig(thinking_budget=THINKING_BUDGET)}
    _err = _probe(_wanted, _m)
    if not _err:
        MODEL_KWARGS[_m] = _wanted
        print(f'  {_m}: thinking budget pinned to {THINKING_BUDGET}')
    else:
        print(f'  WARNING: {_m} rejected thinking_budget={THINKING_BUDGET} — {_err}')
        print('    Falling back to the model default: thinking tokens are UNBOUNDED and can')
        print('    consume MAX_OUTPUT_TOKENS before the answer, producing PARSE_ERROR rows.')
        print('    Watch the measured token report, or use gemini-3.1-flash-lite.')
        _err2 = _probe(config_kwargs, _m)
        assert not _err2, f'{_m} rejected the base config too: {_err2}'
        MODEL_KWARGS[_m] = dict(config_kwargs)

# Default config, used by any cell that does not name a pass (preflight iterates itself).
GEN_CONFIG = types.GenerateContentConfig(**MODEL_KWARGS[ACTIVE_MODELS[0]])
print(f'LLM client ready. Pass models: {PASS_MODELS_RESOLVED}')

## Available Models

Validates `MODEL_NAME` against what this key can actually reach. Quiet when the model is
available; when it is not, it says so and lists the Gemini text models that are — which is
the difference between a two-second fix and a run where every call 404s.

Model availability shifts: `gemini-2.5-flash-lite` was retired for new users on 2026-08-03
and had been this notebook's default.

In [ ]:
%%time
_available = sorted(
    m.name.replace('models/', '')
    for m in client.models.list()
    if not getattr(m, 'supported_actions', None) or 'generateContent' in m.supported_actions
)
_text_models = [
    _n for _n in _available
    if _n.startswith('gemini') and not any(
        _s in _n for _s in ('embedding', 'tts', 'image', 'live', 'native-audio', 'computer-use'))
]
print(f'{len(_available)} models support generateContent; {len(_text_models)} are gemini text models:')
for _n in _text_models:
    print('   ', _n, '   <-- in PASS_MODELS' if _n in ACTIVE_MODELS else '')

# Listing is NECESSARY BUT NOT SUFFICIENT. Observed 2026-08-03: models.list() reported
# gemini-2.5-flash while generate_content on it returned 404 "no longer available to new
# users" — the retirement is enforced at call time, not in the catalogue. So this cell
# cannot confirm a model works; only the Preflight call can. It still catches the clear
# case of a name that is not offered at all.
for _m in ACTIVE_MODELS:
    if _m not in _available:
        print(f"\nWARNING: '{_m}' is not in the catalogue at all — expect a 404.")
    else:
        print(f"\n'{_m}' is listed — but listing does NOT prove it is callable "
              f"(gemini-2.5-flash is listed here and 404s). Preflight is the real check.")

## Prompt Builder

`llm_bootstrap_prompt.md` — the single authored copy of the prompt, sitting beside this
notebook in the repo — embedded verbatim, then split on its fence to recover the exact text
that is sent. Static content first, tweet last, so the fixed prefix stays implicit-cache
eligible.

**The literal below is generated. Do not edit it here.** Edit `llm_bootstrap_prompt.md` and
run `python3 notebooks/05_Classifiers/sync_prompt.py`; `--check` fails while the two differ.

In [ ]:
%%time
# The prompt lives in llm_bootstrap_prompt.md, beside this notebook in the repo. It is
# embedded here verbatim because Colab does not clone the repo at runtime, and it is
# written back out unchanged in the Save section, so the copy that lands next to the
# labels is byte-identical to the repo's.
#
# DO NOT EDIT THE LITERAL BY HAND. Edit llm_bootstrap_prompt.md, then run:
#     python3 notebooks/05_Classifiers/sync_prompt.py
# Hand edits here are silently reverted by the next sync, and `sync_prompt.py --check`
# fails while they are present.
# ── BEGIN GENERATED — llm_bootstrap_prompt.md ──────────────────────────
PROMPT_DOC: str = '''\
# LLM Bootstrap Labelling — Prompt

**This file is the prompt.** The text inside the fence below is sent to the model verbatim,
once per tweet, with `{{TWEET}}` replaced by the tweet being classified and nothing else
added. There is no system instruction, no conversation history, and no context carried
between calls: whatever the model knows about this task, it knows from the fence below.

It is the single authored copy. `notebooks/05_Classifiers/01_llm_bootstrap_labelling.ipynb`
embeds this file verbatim as `PROMPT_DOC` (Colab does not clone the repo at runtime, so it
cannot be read from disk there) and writes it back out unchanged next to the labels it
produces. The copy sitting beside `llm_bootstrap_labels_<stamp>.csv` in Google Drive is byte-identical
to this one — that is asserted, not hoped for.

`categories.md` in this folder explains *why* the taxonomy is shaped this way — what `none`
means, the expected class balance, what has to be decided before adding a category. This file
is what the model is actually told.

## Editing this file

Edit the fence, then run the sync script, then re-run the notebook:

```bash
python3 notebooks/05_Classifiers/sync_prompt.py
```

The script pastes this file into the notebook's `c-prompt` cell and verifies the round-trip.
**Editing the notebook cell by hand instead will be overwritten**, and the sync script fails
loudly if the two have diverged, so the repo can never disagree with itself about what was
sent. Nothing outside the fence reaches the model — the prose in this file is for you and for
whoever reviews the labels.

The tuning loop that this is built for: run with `SMOKE_TEST = True`, open
`llm_bootstrap_labels_full_<stamp>.pkl`, sort by `confidence` ascending and read the bottom rows —
those are where the criteria are underspecified. Edit the fence, sync, re-run.

## The prompt

````text
You are a tweet classifier for a research project on AI public trust.
This is a multi-label classification task about two central questions: "What is art?" and "What makes art valuable?".
The categories come in two groups. `definition` and `value` mark a tweet that raises one of those two questions. The remaining categories are conceptual frameworks that answer them. A tweet can raise a question and answer it through a framework at the same time: label every category that applies.

Classify the tweet into ALL of the following categories that apply:
definition, value, intentionalism, anti_intentionalism, cognitivism, expressivism, hedonism, originality, achievement, none

Per-category criteria:

### definition
Definition: The tweet engages the question of what art is. It claims, denies or asks whether something is or is not art, or whether a person can rightfully be called an artist.
Examples:
- "can we please stop calling it 'ai art' and start calling it 'ai generated images' instead? art requires meaning and intention, ai has none of that" (high: explicitly claims AI art is not art)
- "ai art is not art, it's just garbage 🤷‍♀️" (high: explicitly claims AI art is not art)
- "genuinely curious to know: do you consider ai-generated art to be art or not? please retweet and/or reply with why."
- "if the purpose of the so called 'art' is simply capitalism. money, likes, matterial things them let capitalism efficiency consume art production itself. if ai art isn't art, neither is the marketin' art, as it's just an trade currency."

### value
Definition: The tweet engages the question of what makes art valuable. It judges something as good or bad art, or a good or bad artwork; asks whether a creator can produce anything of value; points out flaws or strengths of a work; or uses the language of liking or disliking a work (this can overlap with personal taste). This is artistic value, NOT monetary value: whether something will sell, or what it is worth in money, does not count.
Examples:
- "cuz if they cleared it up a bit, it would look less like they just posted it to show off or get noticed. this is just me cuz i have seen a fair share of good and bad ai art, yours is in the good category."
- "i don't mind ai art as long as it's not plagiarism...i also think some people a bit too lazy with it, seen some make a few posts with ai art that you can clearly see have flaws. yours is great, i really like it. but i feel like others should touch up the ai art before posting..."

### intentionalism
Definition: This framework posits that the artist's intent is the ultimate source of meaning and the defining characteristic of art. A work is art because it was intended to be so, and it means exactly what the creator meant it to mean. Tweets using this framework will often argue that AI cannot produce art because it lacks a conscious mind, meaning, or deliberate intention.
Example: "can we please stop calling it 'ai art' and start calling it 'ai generated images' instead? art requires meaning and intention, ai has none of that"

### anti_intentionalism
Definition: In direct contrast to intentionalism, this view argues that the author's intent does not govern the meaning of the work. Instead, meaning is derived from the work itself (the language, imagery, or form) and the audience's interpretation. Tweets in this category might invoke the "death of the author" or emphasize that the image speaks for itself regardless of who or what made it.

### cognitivism
Definition: This framework values art for its ability to impart knowledge, foster understanding, or provoke thought. Art is seen as a vehicle for discovering or communicating ideas. Tweets applying this framework will praise art that teaches us something or makes us think, or conversely criticize art for being shallow or intellectually empty.
Example: "gm ☕ i've been playing a lot with ai image generation lately... i like to think i'm a creative person, so for me it's an avenue to express those ideas and a journey of discovery.☄️"

### expressivism
Definition: Expressivism grounds the value (and often the definition) of art in emotion. This can refer to the emotion the artist felt during creation, the emotion captured within the piece, or the feelings it evokes in the audience. Tweets in this category will evaluate works based on their emotional resonance, or dismiss AI art by pointing out its lack of "feeling" or inability to experience emotion.
Example: "🖤 creatures - this ai collection is more about feeling than about a fairy tale🖤"

### hedonism
Definition: This view asserts that the primary value of art lies in the pleasure it brings the viewer. Aesthetic hedonism evaluates art based on whether it is enjoyable, pleasant, or makes the audience feel good. Negative evaluations under this framework will dismiss art simply because it is ugly or unpleasant to look at.

### originality
Definition: Originality values the novel, the creative, and the unprecedented. To be good art (or art at all), the work must offer something non-trivially new. In discourse around AI, this framework is frequently invoked in the negative—criticizing works for being derivative, stolen, traced, copied, or plagiarized from real artists.
Examples:
- "i don't mind ai art as long as it's not plagiarism..."
- "ai is just fancy photoshop for thieves."

### achievement
Definition: This framework ties the value and definition of art to the human effort, skill, and mastery required to produce it. Art is viewed as a triumph of hard work. Tweets using this framework will often attack AI art for being a shortcut, requiring no real skill or patience, and bypassing the arduous learning process that makes traditional art valuable.
Example: "This AI collaboration art thing, isn't effort."

### none
Definition: No category above applies. This is the residual bucket — it is not a claim that the tweet is unrelated to art or to AI. Only output `none` if NO other category applies.

Confidence.
0.8-1.0   Explicit: the vocabulary is present and the link to art's worth/definition is stated outright.
0.5-0.79  Implicit: the appeal is inferred from framing, or shares the tweet with a competing theme.
0.0-0.49  Contested: one plausible reading supports the label, another equally plausible reading does not.
Score confidence for whichever label(s) you chose.

In `rationale`, quote the phrase from the tweet that decided the label.

Return ONLY a JSON array of objects with this exact schema (no prose, no markdown fences). Each object in the array represents a selected category:
[
  {"category": "<one of the categories above>", "confidence": <number between 0 and 1>, "rationale": "<one short sentence>"}
]

Tweet:
"""{{TWEET}}"""
````

## What constrains the reply

The reply shape is enforced **server-side** by the API through a JSON schema, not merely
requested in the prose above. `category` cannot come back as anything outside the label set, and
the reply cannot be wrapped in prose or markdown fences.

```json
{
  "type": "array",
  "items": {
    "type": "object",
    "properties": {
      "category":   {"type": "string", "enum": ["definition", "value", "intentionalism", "anti_intentionalism", "cognitivism", "expressivism", "hedonism", "originality", "achievement", "none"]},
      "confidence": {"type": "number"},
      "rationale":  {"type": "string"}
    },
    "required": ["category", "confidence", "rationale"]
  }
}
```

The enum is generated from the notebook's `CATEGORIES` list rather than copied from here, and
the notebook asserts that list against the category line in the fence above. Adding a category
therefore means editing three things that are checked against each other: `CATEGORIES`, the
category line in the fence, and the criteria describing it.

The notebook also caps output at 512 tokens and pins the thinking budget to 0 where the model
accepts it. If rationales start coming back truncated they surface as `PARSE_ERROR` rows;
raise `MAX_OUTPUT_TOKENS` before suspecting anything else.

## The CSV this produces

`llm_bootstrap_labels_<stamp>.csv`, one row per tweet (`<stamp>` is the run's start date and time), in the same schema as a human
`hitl_review_batch_*.csv`. Since it's multi-label, the CSV contains one-hot columns (0 or 1) for each category.

| Column | Meaning |
| :--- | :--- |
| `id` | Tweet id. |
| `text` | The tweet verbatim. This exact string is what replaced `{{TWEET}}`. |
| `likes`, `retweets` | Engagement counts, for your context only. **Not** part of the prompt. |
| `pred_definition`, `pred_value`, … `pred_none` | The model's binary one-hot prediction for each category (1 if selected, 0 if not). All zeros means the reply could not be parsed (`PARSE_ERROR`). |
| `rationale` | The model's pass-1 rationale(s), one per selected category, separated by a vertical bar. |
| `passes_agree` | Whether every labelling pass chose the same set of categories. |
| `human_definition`, `human_value`, … `human_none` | **Yours.** Empty on delivery — fill in 0 or 1 for every category of a row you review. A row whose `human_*` columns are all filled is used with your labels instead of the model's by notebooks `02` and `03`. |
| `audited` | 1 if a stronger model (`AUDIT_MODEL`, section 6 of the notebook) re-read this row with the same prompt. Only `none`-only rows are audited: the `AUDIT_N` with the most likes + retweets, random fill if fewer have any engagement. |
| `audit_definition`, … `audit_none`, `audit_confidence`, `audit_rationale`, `audit_model` | The audit model's verdict, same shape as `pred_*`. **Blank on rows that were not audited.** A row with `audited = 1` and a substantive `audit_*` label is a false negative of pass 1 in the audit model's judgement — read it first. `pred_*` is never changed by the audit. |

The model's `confidence` and each pass's raw predictions are **not** in the CSV. They are in the
sibling `llm_bootstrap_labels_full_<stamp>.pkl`.

Before overriding a label, read the fence. A label that looks wrong is often the criteria
working exactly as written — which is a reason to edit this file, not just that row.

## How predictions are decided

- The same tweet is labelled **more than once**, independently, each pass through a fresh client.
- **Pass 1 is the operative label** — the one in the CSV. Where passes disagree the row is flagged in `passes_agree` (in the pickle).
- `confidence` in the pickle is the **highest confidence among pass 1's selected categories**. It is the model's own self-report.
'''
PROMPT_DOC_SHA256 = 'c4956915e763'   # of llm_bootstrap_prompt.md at sync time
# ── END GENERATED ─────────────────────────────────────────────────────

# Everything the model receives is inside the single ```` fence; the prose around it is
# for humans and never leaves this notebook. Splitting on the fence rather than on a
# line number means reordering the document cannot silently change the prompt.
_parts = PROMPT_DOC.split('````')
assert len(_parts) == 3, (
    f'expected exactly one four-backtick fence in llm_bootstrap_prompt.md, '
    f'found {max(len(_parts) - 1, 0) // 2}')
PROMPT_TEMPLATE: str = _parts[1].split('\n', 1)[1].rstrip('\n')   # drop the 'text' info string

# The tweet is the only variable part of the prompt, and it goes last so that everything
# before it is byte-identical on every call and stays eligible for the implicit cache.
assert PROMPT_TEMPLATE.count('{{TWEET}}') == 1, (
    'llm_bootstrap_prompt.md must contain exactly one {{TWEET}} placeholder inside the fence')

# CATEGORIES drives the response-schema enum in code; the fence states the label set in
# prose. Neither is derived from the other, so they are checked against each other here —
# a category added to one and not the other is the drift that would otherwise show up as
# the model confidently using a label the schema rejects.
assert ', '.join(CATEGORIES) in PROMPT_TEMPLATE, (
    f'the prompt does not list the label set as CATEGORIES has it: {", ".join(CATEGORIES)!r}')


def build_prompt(tweet_text: str) -> str:
    return PROMPT_TEMPLATE.replace('{{TWEET}}', tweet_text)


print(f'prompt: {len(PROMPT_TEMPLATE):,} chars '
      f'(~{len(PROMPT_TEMPLATE) // 4:,} tokens, re-sent on every call), '
      f'sha256:{PROMPT_DOC_SHA256}')

## Request Builder and Result Parser

One tweet → one JSONL line (`key` = tweet id, `request` = contents + that pass's generation
config); one result line → parsed labels + the `usage_metadata` the API billed. There is no
retry layer: a batch job retries nothing, and a row that fails inside a succeeded job comes
back as an `error` line, which becomes a `PARSE_ERROR` row here. The first failed row of
each job is printed as soon as it is seen.

In [ ]:
%%time
PARSE_ERROR_RESULT = [{'category': 'PARSE_ERROR', 'confidence': 0.0, 'rationale': ''}]

# TOKENOPT_REF.md §16 — the counts the API actually bills, summed from the usage_metadata
# of every result line after download. Rows that failed to parse are included: billed too.
_usage = {'calls': 0, 'input': 0, 'output': 0, 'cached': 0, 'total': 0}


def _reset_usage() -> None:
    _usage.update(calls=0, input=0, output=0, cached=0, total=0)


def _record_usage(meta: dict) -> None:
    if not meta:
        return
    _usage['calls']  += 1
    _usage['input']  += int(meta.get('input', 0) or 0)
    _usage['output'] += int(meta.get('output', 0) or 0)
    _usage['cached'] += int(meta.get('cached', 0) or 0)
    _usage['total']  += int(meta.get('total', 0) or 0)


def usage_cost() -> float:
    return (_usage['input'] / 1e6) * COST_PER_M_INPUT + (_usage['output'] / 1e6) * COST_PER_M_OUTPUT


def gen_config_json(kwargs: dict, **overrides) -> dict:
    """The per-request generation config of one JSONL line: the kwargs the client cell
    probed for that model (schema, thinking budget, output cap) with this pass's overrides
    (temperature), serialised through the SDK's own model so the field names are the ones
    the API accepts. Per-request config is what lets two passes with different
    temperatures — or different models — be separate files with no shared state."""
    cfg = types.GenerateContentConfig(**{**kwargs, **overrides})
    return cfg.model_dump(mode='json', exclude_none=True)


def build_request(key: str, text: str, gen_config: dict) -> dict:
    """One JSONL line. `key` is the tweet id: it is how a result line is matched back to
    its row, so it must be unique within a file. The prompt's static scaffold goes first
    and the tweet last, so every line shares the prefix implicit caching keys on."""
    return {'key': key,
            'request': {'contents': [{'role': 'user', 'parts': [{'text': build_prompt(text)}]}],
                        'generation_config': gen_config}}


def _strip_code_fences(raw: str) -> str:
    raw = raw.strip()
    if raw.startswith('```'):
        raw = re.sub(r'^```(?:json)?\s*', '', raw)
        raw = re.sub(r'\s*```$', '', raw)
    return raw.strip()


def _pick(d, *names, default=None):
    for n in names:
        if isinstance(d, dict) and d.get(n) is not None:
            return d[n]
    return default


def response_text_and_usage(resp) -> tuple[str, dict]:
    """Normalise one response into (text, usage). `resp` is either the dict of a
    downloaded result line (camelCase, as the API serialises it) or the SDK object of an
    inlined response (snake_case attributes). Both shapes are handled because the smoke
    job is inline and the real jobs are files."""
    if not isinstance(resp, dict):
        meta = getattr(resp, 'usage_metadata', None)
        usage = {} if meta is None else {
            'input':  getattr(meta, 'prompt_token_count', 0) or 0,
            'output': getattr(meta, 'candidates_token_count', 0) or 0,
            'cached': getattr(meta, 'cached_content_token_count', 0) or 0,
            'total':  getattr(meta, 'total_token_count', 0) or 0}
        return (getattr(resp, 'text', None) or ''), usage
    text = ''
    cands = resp.get('candidates') or []
    if cands:
        parts = (_pick(cands[0], 'content', default={}) or {}).get('parts') or []
        text = ''.join(p.get('text', '') for p in parts if isinstance(p, dict))
    meta = _pick(resp, 'usageMetadata', 'usage_metadata', default={}) or {}
    usage = {'input':  _pick(meta, 'promptTokenCount', 'prompt_token_count', default=0),
             'output': _pick(meta, 'candidatesTokenCount', 'candidates_token_count', default=0),
             'cached': _pick(meta, 'cachedContentTokenCount', 'cached_content_token_count', default=0),
             'total':  _pick(meta, 'totalTokenCount', 'total_token_count', default=0)}
    return text, usage


def parse_labels(text: str) -> list[dict]:
    """Reply text → [{category, confidence, rationale}, ...]. Raises on anything that is
    not a well-formed array of known categories; the caller turns that into PARSE_ERROR."""
    parsed = json.loads(_strip_code_fences(text or ''))
    if not isinstance(parsed, list):
        raise ValueError(f'Expected array, got {type(parsed)}')
    results = []
    for p in parsed:
        cat = str(p.get('category', '')).strip()
        if cat not in CATEGORIES:
            raise ValueError(f'category {cat!r} not in CATEGORIES')
        results.append({'category': cat,
                        'confidence': float(p.get('confidence', 0.0)),
                        'rationale': str(p.get('rationale', ''))[:500]})
    return results


def parse_results(rows: list, label: str) -> tuple[dict, dict]:
    """Result rows → ({key: labels}, {key: error text}). Every row is inspected — a
    SUCCEEDED job can still carry per-row errors — and every usage is recorded, parse
    errors included. The first failure is printed immediately, not summarised at the end."""
    labels, errors = {}, {}
    first = True
    for row in rows:
        key = str(row['key'])
        if row.get('error'):
            errors[key] = str(row['error'])[:300]
            labels[key] = list(PARSE_ERROR_RESULT)
        else:
            text, usage = response_text_and_usage(row.get('response'))
            _record_usage(usage)
            try:
                labels[key] = parse_labels(text)
            except Exception as e:
                errors[key] = f'{type(e).__name__}: {e} | raw={text[:120]!r}'
                labels[key] = list(PARSE_ERROR_RESULT)
        if key in errors and first:
            print(f'[{label}] first failed row: key={key} — {errors[key]}')
            first = False
    if errors:
        print(f'[{label}] {len(errors):,} of {len(rows):,} rows failed or did not parse → PARSE_ERROR')
    return labels, errors


print('Request builder and result parser ready.')

## Batch Helpers

Upload a file and create a job; poll a set of jobs to a terminal state; fetch a finished job's
result rows (file or inline); read and write the per-run **state file** that holds the job
names. The state file is written the instant a job is created, before anything else can fail —
it is the only handle on a job that is already billed.

In [ ]:
%%time
TERMINAL_STATES = {'JOB_STATE_SUCCEEDED', 'JOB_STATE_FAILED', 'JOB_STATE_CANCELLED',
                   'JOB_STATE_EXPIRED', 'JOB_STATE_PARTIALLY_SUCCEEDED'}
FAILED_STATES   = {'JOB_STATE_FAILED', 'JOB_STATE_CANCELLED', 'JOB_STATE_EXPIRED'}


def _state_name(job) -> str:
    s = getattr(job, 'state', None)
    return getattr(s, 'name', None) or str(s or 'UNKNOWN')


def write_jsonl(path: Path, lines: list) -> Path:
    with open(path, 'w', encoding='utf-8') as f:
        for ln in lines:
            # Compact separators (TOKENOPT_REF.md §1); the file is the request payload.
            f.write(json.dumps(ln, separators=(',', ':'), ensure_ascii=False, default=str) + '\n')
    return path


def count_lines(path: Path) -> int:
    with open(path, 'rb') as f:
        return sum(1 for _ in f)


def load_state() -> dict:
    if BATCH_STATE_JSON.exists():
        with open(BATCH_STATE_JSON) as f:
            return json.load(f)
    return {'run_stamp': RUN_STAMP, 'dataset_type': DATASET_TYPE, 'jobs': {}}


def save_state(state: dict) -> None:
    """Persisted IMMEDIATELY after every create(): the job name is the only handle on a
    job already billed, and resubmitting is a second bill."""
    state['updated'] = time.strftime('%Y-%m-%dT%H:%M:%S')
    with open(BATCH_STATE_JSON, 'w') as f:
        json.dump(state, f, indent=2)


def submit_file_job(model: str, jsonl_path: Path, display_name: str):
    """Upload the request file and create one batch job on it. Returns (job, file name)."""
    uploaded = client.files.upload(
        file=str(jsonl_path),
        config=types.UploadFileConfig(display_name=jsonl_path.name, mime_type='jsonl'))
    job = client.batches.create(model=model, src=uploaded.name,
                                config={'display_name': display_name})
    return job, uploaded.name


def wait_for_jobs(names: list, max_wait_s: float, poll_s: float) -> dict:
    """Poll until every job is terminal or max_wait_s elapses. Returns {name: job}; a job
    that is still running is returned as it is — never cancelled here."""
    t0 = time.time()
    jobs, pending = {}, list(names)
    while pending:
        for n in list(pending):
            j = client.batches.get(name=n)
            jobs[n] = j
            if _state_name(j) in TERMINAL_STATES:
                pending.remove(n)
        if not pending or time.time() - t0 > max_wait_s:
            break
        print(f'  {time.strftime("%H:%M:%S")}  '
              + ', '.join(f'{n.split("/")[-1]}: {_state_name(jobs[n])}' for n in pending)
              + f'  (elapsed {(time.time() - t0) / 60:.0f} min)')
        time.sleep(poll_s)
    return jobs


def fetch_results(job) -> list:
    """The result rows of a terminal job as [{key, response | error}, ...] — from the
    result file for a file job, from the inlined responses for an inline job."""
    dest = getattr(job, 'dest', None)
    if dest is not None and getattr(dest, 'file_name', None):
        raw = client.files.download(file=dest.file_name)
        return [json.loads(ln) for ln in raw.decode('utf-8').splitlines() if ln.strip()]
    rows = []
    for i, r in enumerate(getattr(dest, 'inlined_responses', None) or []):
        err = getattr(r, 'error', None)
        meta = getattr(r, 'metadata', None)
        key = meta.get('key', str(i)) if isinstance(meta, dict) else str(i)
        rows.append({'key': key, 'error': str(err) if err else None,
                     'response': getattr(r, 'response', None)})
    return rows


def describe_job(job) -> str:
    cs = getattr(job, 'completion_stats', None)
    stats = ''
    if cs is not None:
        stats = (f' ok={getattr(cs, "successful_count", None)} '
                 f'failed={getattr(cs, "failed_count", None)} '
                 f'incomplete={getattr(cs, "incomplete_count", None)}')
    err = getattr(job, 'error', None)
    return f'{job.name}: {_state_name(job)}{stats}' + (f' error={err}' if err else '')


print('Batch helpers ready.')

# 2 · Preflight

**Run this before submitting anything.** Two checks, cheapest first:

1. One **synchronous** request per model with the real prompt and real config — it reports in
   two seconds what a batch job would take hours to report (a retired model, a rejected
   schema, a bad key).
2. A **two-request inline batch job** per model — the only thing that proves the batch path
   itself: that the model accepts batch jobs at all (support is per model and the catalogue
   over-reports), that the per-request config survives the file format, and that a result
   line parses.

## Connection Check — One Call

In [ ]:
%%time
# The Run section refuses to start unless this cell has set PREFLIGHT_OK = True.
PREFLIGHT_OK = False
_probe_text = 'ai art is just theft, it copies real artists and calls it new'

# EVERY distinct model is preflighted, not just the first. A model that is only reached
# in pass 2 would otherwise fail for the first time after pass 1 has already been paid
# for — and a 404 or a rejected config fails identically on every subsequent row.
try:
    for _pm in ACTIVE_MODELS:
        print(f'--- {_pm} ---')
        _r = client.models.generate_content(
            model=_pm, contents=build_prompt(_probe_text),
            config=types.GenerateContentConfig(**MODEL_KWARGS[_pm]),
        )
        print('REQUEST OK')
        print('  raw response :', (_r.text or '')[:300])
        _m = getattr(_r, 'usage_metadata', None)
        if _m is not None:
            print(f'  tokens       : input={_m.prompt_token_count}, '
                  f'output={_m.candidates_token_count}, '
                  f'cached={getattr(_m, "cached_content_token_count", 0)}')
        else:
            print('  tokens       : usage_metadata absent on this response')
        _parsed = json.loads(_strip_code_fences(_r.text or ''))
        for p in _parsed:
            assert p['category'] in CATEGORIES, f'category {p["category"]!r} not in CATEGORIES'
        print(f'  parsed       : {_parsed}')
    PREFLIGHT_OK = True
    print(f'\nPreflight passed for all {len(ACTIVE_MODELS)} model(s) — safe to run the loop.')
except Exception as _e:
    print('REQUEST FAILED — do NOT run the labelling loop yet.')
    print(f'  {type(_e).__name__}: {_e}')
    print('\nCommon causes:')
    print('  * INVALID_ARGUMENT naming response_json_schema / response_schema')
    print('       -> set USE_RESPONSE_SCHEMA = False in Configuration, re-run the client cell.')
    print('  * NOT_FOUND \'no longer available to new users\'')
    print('       -> that model is closed to new accounts. Re-run the Available Models cell')
    print('          and pick a listed model from a NEWER family (the whole gemini-2.5 line')
    print('          was closed to new users on 2026-08-03), then re-run Configuration.')
    print('  * 401 / API key not valid')
    print('       -> the Colab secret holds a Vertex-AI key; mint an AI Studio key instead.')
    raise

## Batch Smoke Test — Inline Job

Waits up to `BATCH_SMOKE_WAIT_S`; a job that has not finished by then is **cancelled** (so it
cannot bill later) and the run continues with a warning. A job that finishes in any state other
than `SUCCEEDED`, or whose rows fail, stops the run here.

In [ ]:
%%time
BATCH_SMOKE_OK = False
if not BATCH_SMOKE:
    print('BATCH_SMOKE is False — skipping the inline smoke job. The file jobs will be the '
          'first test of the batch path.')
elif not globals().get('PREFLIGHT_OK', False):
    raise RuntimeError('The synchronous preflight has not passed; fix that first.')
else:
    _smoke_texts = [_probe_text,
                    'honestly the new model just makes pretty pictures, nothing more'][:BATCH_SMOKE_N]
    _smoke_jobs = {}
    for _pm in ACTIVE_MODELS:
        _reqs = [{'contents': [{'role': 'user', 'parts': [{'text': build_prompt(_t)}]}],
                  'config': gen_config_json(MODEL_KWARGS[_pm]),
                  'metadata': {'key': f'smoke-{_i}'}}
                 for _i, _t in enumerate(_smoke_texts)]
        _job = client.batches.create(model=_pm, src=_reqs,
                                     config={'display_name': f'llm_bootstrap_smoke_{RUN_STAMP}_{_pm}'})
        _smoke_jobs[_job.name] = _pm
        print(f'{_pm}: inline smoke job {_job.name} ({len(_reqs)} requests) → {_state_name(_job)}')
    _done = wait_for_jobs(list(_smoke_jobs), BATCH_SMOKE_WAIT_S, min(BATCH_POLL_S, 30))
    _all_ok = True
    for _n, _pm in _smoke_jobs.items():
        _j = _done[_n]
        print(describe_job(_j))
        if _state_name(_j) not in TERMINAL_STATES:
            client.batches.cancel(name=_n)
            print(f'  not finished within {BATCH_SMOKE_WAIT_S}s — CANCELLED. The batch queue is slow '
                  'right now; the file jobs will still be submitted, but expect a long wait.')
            _all_ok = False
            continue
        if _state_name(_j) != 'JOB_STATE_SUCCEEDED':
            raise RuntimeError(f'{_pm}: inline smoke job ended {_state_name(_j)}: '
                               f'{getattr(_j, "error", None)} — do not submit the file jobs.')
        _labels, _errs = parse_results(fetch_results(_j), f'smoke {_pm}')
        for _k, _v in _labels.items():
            print(f'  {_k}: {_v}')
        if _errs:
            raise RuntimeError(f'{_pm}: smoke rows failed — {_errs}')
    BATCH_SMOKE_OK = _all_ok
    print('\nBatch smoke ' + ('PASSED' if _all_ok else 'INCONCLUSIVE (timed out)')
          + f' for {len(_smoke_jobs)} model(s); '
          + f"{_usage['calls']} result rows, {_usage['input']:,} in / {_usage['output']:,} out tokens.")

# 3 · Load and Estimate

Load the partition, apply the deterministic nested selection and the row cap, then project
token cost **before** any spend.

## Load Input

In [ ]:
%%time
assert INPUT_PATH.exists(), f'Input not found: {INPUT_PATH}. Run 00_hitl_data_preparation.ipynb first.'
df = pd.read_pickle(INPUT_PATH)
print(f'Loaded {len(df):,} tweets from {INPUT_PATH.name}')

for col in ('id', 'text', 'likes', 'retweets'):
    if col not in df.columns:
        df[col] = '' if col in ('id', 'text') else 0

df['text'] = df['text'].astype(str)
df['id'] = df['id'].astype(str)

# Deterministic, NESTED selection. Sort by id first so the permutation depends only on
# WHICH tweets are in the partition, not on the order the pickle happens to store them
# in; then take a head slice. head(100) is therefore a strict subset of head(1000), so a
# smoke run and a later full run label overlapping tweets and the basket below grows
# monotonically. (Two `df.sample(n=...)` calls sharing a seed but differing in n do NOT
# nest — they would have produced two largely disjoint sets.)
df = df.sort_values('id').sample(frac=1, random_state=SELECTION_SEED).reset_index(drop=True)

n_target = min(SMOKE_TEST_N if SMOKE_TEST else len(df), MAX_LLM_TWEETS)
df = df.head(n_target).reset_index(drop=True)

assert len(df) <= MAX_LLM_TWEETS, f'{len(df):,} rows exceeds MAX_LLM_TWEETS ({MAX_LLM_TWEETS:,})'
print(f'Selection: {"SMOKE_TEST" if SMOKE_TEST else "full"} → first {len(df):,} of the fixed '
      f'permutation (seed {SELECTION_SEED}); MAX_LLM_TWEETS={MAX_LLM_TWEETS:,}')
print(f'Will send {len(df):,} tweets to {MODEL_NAME}')

## Token and Cost Estimate

`TOKENOPT_REF.md` §17. Built from the real prompts for the rows actually loaded, so it tracks
the current criteria. Output is an upper bound (assumes every response fills
`MAX_OUTPUT_TOKENS`); the measured figures after collection are authoritative. Cost is at the
**batch rate** (`BATCH_RATE_FACTOR` x the interactive rate). In batch mode this estimate is
also the token ceiling's enforcement point — see the next section.

In [ ]:
%%time
_prompts     = [build_prompt(t) for t in df['text']]
_chars       = sum(len(p) for p in _prompts)
_static      = len(build_prompt(''))
est_input    = _chars / CHARS_PER_TOKEN
est_output   = len(df) * MAX_OUTPUT_TOKENS
_passes      = max(N_LABEL_PASSES, 1)
est_cost     = (est_input / 1e6 * COST_PER_M_INPUT
                + est_output / 1e6 * COST_PER_M_OUTPUT) * max(N_LABEL_PASSES, 1)

print(f'{len(df):,} tweets x {_passes} pass(es) = {len(df) * _passes:,} batch requests; '
      f'pass models {PASS_MODELS_RESOLVED}')
print(f'  est. input  : {est_input:>12,.0f} tokens')
print(f'  est. output : {est_output:>12,.0f} tokens  (upper bound: MAX_OUTPUT_TOKENS each)')
print(f'  est. cost   : ${est_cost:>11,.4f}  @ batch rate ${COST_PER_M_INPUT}/M in, '
      f'${COST_PER_M_OUTPUT}/M out'
      + ('  (dearest model\'s rates — UPPER BOUND)' if RATES_ARE_WORST_CASE else ''))
print(f'  static scaffold: {_static:,} of {_chars/max(len(df),1):,.0f} chars per prompt '
      f'({_static/(_chars/max(len(df),1)):.0%}) — identical every call, so implicit-cache eligible')

if (est_input + est_output) * _passes > MAX_SESSION_TOKENS:
    print(f'\n*** MAX_SESSION_TOKENS ({MAX_SESSION_TOKENS:,}) is BELOW this run\'s upper-bound '
          f'estimate of {(est_input + est_output) * _passes:,.0f} tokens over {_passes} pass(es).')
    print('    The Build Request Files cell will REFUSE to write the files: a batch job cannot')
    print('    be stopped partway, so the ceiling is enforced before anything is uploaded. ***')

if est_cost > COST_ALERT_USD:
    print(f'\n*** COST ALERT: ${est_cost:,.2f} exceeds COST_ALERT_USD (${COST_ALERT_USD:,.2f}). '
          f'Review before running the next cell. ***')

del _prompts   # a full copy of every prompt; no reason to hold it through the run

# 4 · Run — Batch API

Three cells, each re-runnable and each a no-op when its work is already on record in the
state file:

1. **Build Request Files** — one JSONL per pass under `Classifiers_Data/HITL/batch/`. The
   row cap is the line count; the token ceiling is checked against the upper-bound estimate.
   Nothing here talks to the API.
2. **Submit Jobs** — upload each file, create one job per pass, write the job name to the state
   file before doing anything else.
3. **Wait and Collect** — poll to a terminal state, download and persist the result files (the
   API deletes them after six weeks), parse every line, report the billed usage, and build
   `out_df` in exactly the shape `01` builds it.

If polling outlives `BATCH_WAIT_MAX_S`, step 3 stops with the stamp to put in
`RESUME_RUN_STAMP`; the next session skips 1 and 2 and collects.

## Build Request Files

In [ ]:
%%time
if not globals().get('PREFLIGHT_OK', False):
    raise RuntimeError(
        'Preflight has not passed. Run the Preflight section (2) and fix what it reports '
        'before building anything. Override only if you know why: PREFLIGHT_OK = True.')

assert len(df) <= MAX_LLM_TWEETS, f'{len(df):,} rows exceeds MAX_LLM_TWEETS ({MAX_LLM_TWEETS:,})'
_temps = PASS_TEMPERATURES or [TEMPERATURE] * N_LABEL_PASSES
assert len(_temps) >= N_LABEL_PASSES, (
    f'PASS_TEMPERATURES has {len(_temps)} entries but N_LABEL_PASSES is {N_LABEL_PASSES}')

# Token ceiling — pre-flight only. A batch job cannot be stopped at a token count, so the
# upper bound (every reply filling MAX_OUTPUT_TOKENS, every pass) is checked here and the
# files are not written if it fails. Nothing has been uploaded or billed at this point.
_ub = (est_input + est_output) * max(N_LABEL_PASSES, 1)
if _ub > MAX_SESSION_TOKENS:
    raise RuntimeError(
        f'Upper-bound estimate {_ub:,.0f} tokens for {N_LABEL_PASSES} pass(es) exceeds '
        f'MAX_SESSION_TOKENS ({MAX_SESSION_TOKENS:,}). Nothing written. Raise the ceiling '
        'deliberately in Configuration, or lower MAX_LLM_TWEETS.')

state = load_state()
if state['jobs']:
    # Resuming: the files were written at submit time. The selection reproduced now must be
    # the one that was submitted, or the results would be matched to the wrong rows.
    assert state['ids'] == list(df['id']), (
        'The selection reproduced now differs from the one submitted — SELECTION_SEED, '
        'SMOKE_TEST, MAX_LLM_TWEETS or the input changed. Do not collect these jobs into this '
        'dataframe; restore the Configuration the run was submitted with.')
    print(f'Resuming {RUN_STAMP}: {len(state["jobs"])} job(s) on record; request files kept.')
else:
    state['ids'] = list(df['id'])
    state['pass_models'] = PASS_MODELS_RESOLVED
    state['pass_temperatures'] = _temps
    state['prompt_doc_sha256'] = PROMPT_DOC_SHA256
    state['request_files'] = {}
    for _p in range(N_LABEL_PASSES):
        _pm   = PASS_MODELS_RESOLVED[_p]
        _cfg  = gen_config_json(MODEL_KWARGS[_pm], temperature=_temps[_p])
        _path = BATCH_FOLDER / f'llm_bootstrap_batch_{RUN_STAMP}_pass{_p + 1}.jsonl'
        write_jsonl(_path, [build_request(row['id'], row['text'], _cfg) for _, row in df.iterrows()])
        _n = count_lines(_path)
        # Third layer of the row cap. The file is what gets billed, so the file is what is
        # counted — not the dataframe it was built from.
        assert _n == len(df) and _n <= MAX_LLM_TWEETS, (
            f'{_path.name}: {_n:,} lines (expected {len(df):,}, cap {MAX_LLM_TWEETS:,})')
        state['request_files'][f'pass{_p + 1}'] = _path.name
        print(f'pass {_p + 1}: {_path.name} — {_n:,} requests, {_path.stat().st_size / 1e6:.1f} MB, '
              f'{_pm} @ T={_temps[_p]}')
    save_state(state)
print(f'{N_LABEL_PASSES} pass(es) x {len(df):,} tweets = {N_LABEL_PASSES * len(df):,} requests; '
      f'state → {BATCH_STATE_JSON.name}')

## Submit Jobs

In [ ]:
%%time
state = load_state()
if state.get('jobs'):
    print('Jobs already submitted for this run — nothing to do:')
    for _k, _v in state['jobs'].items():
        print(f'  {_k}: {_v["name"]} ({_v.get("requests", "?")} requests, {_v["model"]})')
else:
    assert state.get('request_files'), 'No request files on record — run Build Request Files first.'
    for _p in range(N_LABEL_PASSES):
        _key  = f'pass{_p + 1}'
        _pm   = PASS_MODELS_RESOLVED[_p]
        _path = BATCH_FOLDER / state['request_files'][_key]
        _n    = count_lines(_path)
        assert _n <= MAX_LLM_TWEETS, f'{_path.name}: {_n:,} lines exceeds MAX_LLM_TWEETS'
        _job, _file = submit_file_job(_pm, _path, f'llm_bootstrap_{RUN_STAMP}_{_key}')
        state['jobs'][_key] = {'name': _job.name, 'model': _pm, 'temperature': _temps[_p],
                               'input_file': _file, 'requests': _n,
                               'submitted': time.strftime('%Y-%m-%dT%H:%M:%S')}
        save_state(state)          # before anything else can fail
        print(f'{_key}: {describe_job(_job)}')
    print(f'\nSubmitted {len(state["jobs"])} job(s). To collect in a fresh session, set\n'
          f'    RESUME_RUN_STAMP = {RUN_STAMP!r}\n'
          'in Configuration and run the notebook from the top.')

## Wait and Collect

In [ ]:
%%time
state = load_state()
assert state.get('jobs'), 'No jobs on record for this run — run the Submit cell first.'
_names = [v['name'] for v in state['jobs'].values() if not v.get('collected')]
if not _names:
    _names = [v['name'] for v in state['jobs'].values()]   # re-collect: cheap, and re-runnable
print(f'Waiting on {len(_names)} job(s), polling every {BATCH_POLL_S}s for up to '
      f'{BATCH_WAIT_MAX_S / 3600:.1f} h ...')
t0 = time.time()
jobs = wait_for_jobs(_names, BATCH_WAIT_MAX_S, BATCH_POLL_S)
for _k, _v in state['jobs'].items():
    if _v['name'] in jobs:
        print(f'  {_k}: {describe_job(jobs[_v["name"]])}')

_unfinished = [k for k, v in state['jobs'].items()
               if v['name'] in jobs and _state_name(jobs[v['name']]) not in TERMINAL_STATES]
if _unfinished:
    raise RuntimeError(
        f'{_unfinished} still running after {BATCH_WAIT_MAX_S / 60:.0f} min. Nothing is lost: set '
        f'RESUME_RUN_STAMP = {RUN_STAMP!r} in Configuration and re-run the notebook from the '
        'top later. Jobs expire 48 h after submission.')

_reset_usage()
stopped_early = False
passes: list[dict] = []          # pass index -> {tweet id: classification list}
for _p in range(N_LABEL_PASSES):
    _k  = f'pass{_p + 1}'
    _j  = jobs[state['jobs'][_k]['name']]
    _st = _state_name(_j)
    if _st in FAILED_STATES:
        stopped_early = True
        print(f'*** {_k} ended {_st}: {getattr(_j, "error", None)} — no rows from this pass. ***')
        break
    _rows = fetch_results(_j)
    # Persist the raw result file beside the request file: the API deletes results after
    # six weeks, and the usage figures below are only reproducible from this copy.
    _out = BATCH_FOLDER / f'llm_bootstrap_batch_{RUN_STAMP}_{_k}_results.jsonl'
    write_jsonl(_out, _rows)
    _labels, _errs = parse_results(_rows, _k)
    _missing = set(df['id']) - set(_labels)
    if _missing:
        print(f'  {_k}: {len(_missing):,} tweet(s) have no result line at all (job {_st}).')
    passes.append(_labels)
    state['jobs'][_k].update({'state': _st, 'result_file': _out.name, 'rows': len(_rows),
                              'failed_rows': len(_errs),
                              'collected': time.strftime('%Y-%m-%dT%H:%M:%S')})
    save_state(state)
    print(f'  {_k}: {len(_rows):,} result rows → {_out.name}')

# Every tweet the LLM may have READ. The uploaded file is the record of what was sent
# (LLM_BATCH_JOB_SKILL.md): once any pass's job ran, every id in the file is training
# data, whether or not its result line came back or parsed. Computed before the
# completeness filter below, so rows a failed pass dropped from out_df are still basketed
# — releasing them back into the held-out pool is the leak the basket exists to prevent.
sent_ids = sorted(set(df['id'])) if passes else []

# Keep only tweets that every pass managed to label, so agreement is computed on a
# complete matrix rather than a ragged one.
_complete = [i for i in df['id'] if all(i in p for p in passes)]
if len(_complete) < len(df):
    print(f'\n{len(df) - len(_complete):,} tweet(s) missing from at least one pass — excluded '
          f'from out_df, but still basketed as seen ({len(sent_ids):,} sent in total).')

results = []
for _, row in df[df['id'].isin(_complete)].iterrows():
    _labels = [sorted([x['category'] for x in passes[p][row['id']]]) for p in range(len(passes))]
    _agree  = all(tuple(l) == tuple(_labels[0]) for l in _labels)

    # Pass 1 is operative
    operative_preds = passes[0][row['id']]
    op_cats = [x['category'] for x in operative_preds]

    rec = {
        'id': row['id'],
        'text': row['text'],
        'likes': row.get('likes', 0),
        'retweets': row.get('retweets', 0),
        'passes_agree': _agree,
        'confidence': max([float(x.get('confidence', 0.0)) for x in operative_preds]) if operative_preds else 0.0,
        'rationale': " | ".join([str(x.get('rationale', '')) for x in operative_preds]) if operative_preds else '',
    }

    # one-hot pred and human label columns
    for cat in CATEGORIES:
        if cat == 'none': continue
        rec[f'pred_{cat}'] = 1 if cat in op_cats else 0
        rec[f'human_{cat}'] = ''
    rec['pred_none'] = 1 if 'none' in op_cats else 0
    rec['human_none'] = ''

    for p in range(len(passes)):
        rec[f'predictions_pass{p + 1}'] = passes[p][row['id']]
    rec['parse_error'] = 1 if sum([rec.get(f'pred_{c}', 0) for c in CATEGORIES]) == 0 else 0
    results.append(rec)

out_df = pd.DataFrame(results)
print(f'\n{"STOPPED EARLY" if stopped_early else "Done"}. '
      f'{len(out_df):,} of {len(df):,} tweets labelled by {len(passes)} pass(es). '
      f'Wait + collect time: {time.time()-t0:.1f}s')
print('Prediction sums:')
for cat in CATEGORIES:
    if f'pred_{cat}' in out_df:
        print(f'{cat}: {out_df[f"pred_{cat}"].sum()}')

if len(passes) > 1 and len(out_df):
    _n_dis = int((~out_df['passes_agree']).sum())
    print(f"\nInter-pass agreement: {1 - _n_dis / len(out_df):.1%} "
          f"({_n_dis:,} disagreement(s) of {len(out_df):,}) — see the Review section.")

# TOKENOPT_REF.md §16 — measured, not estimated.
if _usage['calls']:
    print(f"\nMeasured usage over {_usage['calls']:,} result rows (batch rate):")
    print(f"  input   : {_usage['input']:>12,} tokens")
    print(f"  cached  : {_usage['cached']:>12,} tokens "
          f"({_usage['cached']/max(_usage['input'],1):.1%} of input — implicit cache hit rate)")
    print(f"  output  : {_usage['output']:>12,} tokens")
    print(f"  total   : {_usage['total']:>12,} tokens")
    _est = globals().get('est_cost')
    _cmp = (f'  (est. was ${_est * len(passes):,.4f} for {len(passes)} passes)'
            if _est is not None else '  (no pre-flight estimate)')
    print(f"  cost    : ${usage_cost():,.4f}{_cmp}")
else:
    print('\nNo usage_metadata in the result lines — spend cannot be measured from them.')

# 5 · Review

Eyeball the labelling before it becomes training data. Distribution, confidence, a cue-word
audit, and worked examples — enough to judge whether `categories.md` is doing what you meant.

Nothing here calls the API or writes to disk, so it is free to re-run.

## Classification Report

In [ ]:
%%time
# ─── Knobs for this report ───────────────────────────────────────────────
LOW_CONF_BELOW = 0.8
N_POSITIVE = 100
N_SAMPLES  = 10
# ────────────────────────────────────────────────────────────────────────

import textwrap

# Extract cue words logic
CUE_WORDS = [
    'steal', 'stole', 'stolen', 'theft', 'thief', 'plagiaris', 'copy',
    'scrap', 'consent', 'permission', 'compensat', 'credit', 'dataset',
    'real art', 'human art', 'soul', 'effort', 'talent', 'lazy',
    'luddite', 'gatekeep', 'adapt', 'tool', 'prompt', 'democratiz',
]

def _has_cue(text: str) -> bool:
    low = str(text).lower()
    return any(w in low for w in CUE_WORDS)

def _show(rows, title: str) -> None:
    print(f'\n{title}')
    print('-' * 80)
    if len(rows) == 0:
        print('  (none)')
        return
    for _, r in rows.iterrows():
        cue = 'cue' if _has_cue(r['text']) else 'no-cue'
        # Get active categories
        active_cats = [c for c in CATEGORIES if r.get(f'pred_{c}', 0) == 1]
        print(f"  [{', '.join(active_cats)}]  conf={r.get('confidence',0):.2f}  {cue}"
              f"  likes={r.get('likes', 0)}  rt={r.get('retweets', 0)}")
        for line in textwrap.wrap(str(r['text']), 80 - 6) or ['']:
            print(f'      {line}')
        for line in textwrap.wrap(f"why: {r.get('rationale','')}", 80 - 6):
            print(f'      | {line}')
        print()

out_df['_cue'] = out_df['text'].map(_has_cue)

# We added 'parse_error' in out_df or we can deduce it if no category is 1.
out_df['parse_error'] = (out_df[[f'pred_{c}' for c in CATEGORIES]].sum(axis=1) == 0).astype(int)
ok = out_df[out_df['parse_error'] == 0].copy()
if 'confidence' not in ok:
    ok['confidence'] = 0.0
ok['confidence'] = pd.to_numeric(ok['confidence'], errors='coerce').fillna(0.0)

print('=' * 80)
print(f'CLASSIFICATION REPORT — {len(out_df):,} rows')
print('=' * 80)

# ─── 1. Label distribution ───────────────────────────────────────────────
print('\nLABEL DISTRIBUTION (Multi-label)')
print('-' * 80)
for cat in CATEGORIES:
    n = int(ok[f'pred_{cat}'].sum()) if f'pred_{cat}' in ok else 0
    bar = '#' * int(40 * n / max(len(out_df), 1))
    print(f'  {cat:<20} {n:>6,}  {n/len(out_df):>6.1%}  {bar}')

n_err = int(out_df['parse_error'].sum())
if n_err:
    print(f'\n  WARNING: {n_err:,} PARSE_ERROR rows are excluded from everything below.')

# ─── 2. Confidence ───────────────────────────────────────────────────────
print('\nCONFIDENCE BY LABEL')
print('-' * 80)
print(f"  {'label':<20} {'n':>6} {'mean':>7} {'median':>7} {'min':>6} {'< ' + str(LOW_CONF_BELOW):>9}")
for cat in CATEGORIES:
    if f'pred_{cat}' in ok:
        grp = ok[ok[f'pred_{cat}'] == 1]
        if len(grp) > 0:
            print(f"  {cat:<20} {len(grp):>6} {grp['confidence'].mean():>7.2f} "
                  f"{grp['confidence'].median():>7.2f} {grp['confidence'].min():>6.2f} "
                  f"{(grp['confidence'] < LOW_CONF_BELOW).sum():>9}")

# ─── 3. Samples ──────────────────────────────────────────────────────────
for label in CATEGORIES:
    if f'pred_{label}' in ok:
        grp = ok[ok[f'pred_{label}'] == 1]
        _show(grp.sample(n=min(N_SAMPLES, len(grp)), random_state=42),
              f'RANDOM SAMPLE — {label} ({len(grp):,} rows)')


## Label Co-occurrence

Multi-label output, so the distribution above says nothing about which labels travel
together. Three views of the same pairwise matrix, restricted to rows carrying at least one
substantive label (`none`-only rows carry no pairing information): raw counts, the
conditional share `P(column | row)`, and the phi coefficient (Pearson correlation of the
0/1 columns). The last block splits the rows by whether they carry a *question* label
(`definition`, `value`), a *framework* label, or both — the taxonomy expects frameworks to
answer a question that the same tweet raises.

In [ ]:
%%time
# ─── 4. Label co-occurrence ──────────────────────────────────────────────
_subst = [c for c in CATEGORIES if c != 'none']
_L = ok[[f'pred_{c}' for c in CATEGORIES]].astype(int)
_L.columns = CATEGORIES
_S = _L.loc[_L[_subst].sum(axis=1) > 0, _subst]
_none_plus = int(((_L['none'] == 1) & (_L[_subst].sum(axis=1) > 0)).sum())

print('\nLABEL CO-OCCURRENCE')
print('-' * 80)
print(f'  {len(_S):,} of {len(ok):,} rows carry a substantive label; labels per such row: '
      + ', '.join(f'{k}: {v:,}' for k, v in _S.sum(axis=1).value_counts().sort_index().items()))
if _none_plus:
    print(f'  WARNING: {_none_plus:,} row(s) carry `none` together with another label; '
          'the prompt says `none` is only for rows where nothing else applies.')

_co = _S.T.dot(_S)
with pd.option_context('display.width', 200, 'display.max_columns', None):
    print('\n  Pairwise counts (diagonal = rows carrying that label):')
    print(_co.to_string())
    print('\n  P(column | row) — share of rows with the row label that also carry the column label:')
    print(_co.div(pd.Series(np.diag(_co), index=_co.index).replace(0, np.nan), axis=0)
          .round(2).to_string())
    print('\n  Phi coefficient (+1 = always together, -1 = never together, NaN = label unused):')
    print(_S.corr().round(2).to_string())

_q  = [c for c in ('definition', 'value') if c in _subst]
_fw = [c for c in _subst if c not in _q]
_has_q, _has_fw = _S[_q].sum(axis=1) > 0, _S[_fw].sum(axis=1) > 0
_fw_only = _S.loc[~_has_q & _has_fw, _fw].sum()
print('\n  Question labels (definition / value) against framework labels:')
print(f'    question only  : {int((_has_q & ~_has_fw).sum()):>6,}')
print(f'    both           : {int((_has_q & _has_fw).sum()):>6,}')
print(f'    framework only : {int((~_has_q & _has_fw).sum()):>6,}  '
      f'{_fw_only[_fw_only > 0].to_dict()}')

## Pass Disagreement Study

`passes_agree` says *that* the deterministic pass and the sampled pass differ; this says *how*.
Every disagreement is decomposed into labels pass 2 **dropped** and labels it **added**, tallied
per label against that label's pass-1 prevalence, then grouped into kinds (a `none` row gaining a
label, a labelled row collapsing to `none`, a swap, an addition, a removal). Pass-1 confidence is
compared between agreeing and disagreeing rows, and the examples show both passes' labels,
confidences and quoted phrases side by side, lowest pass-1 confidence first — those are the rows
where the criteria under-determine the answer.

In [ ]:
%%time
# ─── 5. Pass disagreement study ──────────────────────────────────────────
N_DISAGREE_SAMPLES = 15
_pass_cols = sorted(c for c in ok.columns if c.startswith('predictions_pass'))
if len(_pass_cols) < 2:
    print('\nPASS DISAGREEMENT STUDY: single-pass run, nothing to compare.')
else:
    def _labset(preds): return frozenset(x['category'] for x in (preds or []))
    def _fmt(preds):
        return '; '.join(f"{x['category']} ({float(x.get('confidence', 0)):.2f}) "
                         f"\"{str(x.get('rationale', ''))[:90]}\"" for x in (preds or [])) or '(empty)'
    _p1, _p2 = _pass_cols[0], _pass_cols[1]
    _dis = ok[~ok['passes_agree'].astype(bool)].copy()
    _dis['_s1'] = _dis[_p1].map(_labset); _dis['_s2'] = _dis[_p2].map(_labset)
    _dis['_dropped'] = [a - b for a, b in zip(_dis['_s1'], _dis['_s2'])]
    _dis['_added']   = [b - a for a, b in zip(_dis['_s1'], _dis['_s2'])]
    def _kind(r):
        s1, s2 = r['_s1'], r['_s2']
        if s1 == {'none'} and s2 != {'none'}: return 'none -> label'
        if s2 == {'none'} and s1 != {'none'}: return 'label -> none'
        if r['_dropped'] and r['_added']:     return 'swap'
        if r['_added']:                       return 'label added'
        return 'label dropped'
    _dis['_kind'] = _dis.apply(_kind, axis=1)

    print('\nPASS DISAGREEMENT STUDY')
    print('-' * 80)
    print(f"  {len(_dis):,} of {len(ok):,} rows disagree ({len(_dis) / max(len(ok), 1):.1%}); "
          f"pass 1 = {_p1[-5:]}, pass 2 = {_p2[-5:]}")
    print('\n  Kind of disagreement:')
    for k, v in _dis['_kind'].value_counts().items():
        print(f'    {k:<16} {v:>5,}')
    print(f"\n  {'label':<20} {'pass-1 rows':>11} {'dropped by 2':>13} {'added by 2':>11} {'churn':>7}")
    for cat in CATEGORIES:
        n1 = int(ok[f'pred_{cat}'].sum())
        nd = int(sum(cat in d for d in _dis['_dropped']))
        na = int(sum(cat in a for a in _dis['_added']))
        print(f"  {cat:<20} {n1:>11,} {nd:>13,} {na:>11,} {(nd + na) / n1 if n1 else float('nan'):>7.1%}")
    print('\n  Pass-1 confidence (max over selected labels):')
    for name, grp in (('agree', ok[ok['passes_agree'].astype(bool)]), ('disagree', _dis)):
        if len(grp):
            print(f"    {name:<9} n={len(grp):>5,}  mean={grp['confidence'].mean():.2f}  "
                  f"median={grp['confidence'].median():.2f}  "
                  f"< {LOW_CONF_BELOW}: {(grp['confidence'] < LOW_CONF_BELOW).mean():.0%}")
    print(f'\n  EXAMPLES — {min(N_DISAGREE_SAMPLES, len(_dis))} disagreeing rows, lowest pass-1 confidence first')
    print('-' * 80)
    for _, r in _dis.sort_values('confidence').head(N_DISAGREE_SAMPLES).iterrows():
        print(f"  [{r['_kind']}]  likes={r.get('likes', 0)}  rt={r.get('retweets', 0)}")
        for line in textwrap.wrap(str(r['text']), 80 - 6) or ['']:
            print(f'      {line}')
        for line in textwrap.wrap('pass 1: ' + _fmt(r[_p1]), 80 - 6):
            print(f'      | {line}')
        for line in textwrap.wrap('pass 2: ' + _fmt(r[_p2]), 80 - 6):
            print(f'      | {line}')
        print()

# 6 · Second Opinion on `none`

The positive labels have been spot-checked and show almost no false positives. The false
negatives have not been checked at all, and `none` is most of the output. This section has a
stronger model (`AUDIT_MODEL`) re-read a **fixed** number of `none`-only rows — the `AUDIT_N`
with the most likes + retweets, topped up at random when fewer than `AUDIT_N` have any
engagement — with the **same prompt**. Its verdict goes into `audited` / `audit_<label>` /
`audit_confidence` / `audit_rationale` beside the operative columns; `pred_*` is never changed.
A row where the two models disagree is the highest-value row a human can read.

It runs as a third batch job, resumable like the others. Cost is bounded two ways, both
before upload: `AUDIT_N` rows as the line count of the audit file, and `AUDIT_MAX_TOKENS`
against the upper-bound estimate. Set `AUDIT_ENABLED = False` to skip the
section entirely (the Save cell then writes no audit columns).

## Select the Rows and Estimate the Cost

In [ ]:
%%time
audit_df = ok.iloc[0:0].copy()
if not AUDIT_ENABLED:
    print('AUDIT_ENABLED is False — section skipped.')
else:
    _subst_cols = [f'pred_{c}' for c in CATEGORIES if c != 'none']
    _pool = ok[(ok['pred_none'] == 1) & (ok[_subst_cols].sum(axis=1) == 0)].copy()
    _pool['_engagement'] = (pd.to_numeric(_pool['likes'], errors='coerce').fillna(0)
                            + pd.to_numeric(_pool['retweets'], errors='coerce').fillna(0))
    _engaged = _pool[_pool['_engagement'] > 0].sort_values(['_engagement', 'id'], ascending=[False, True])
    _top = _engaged.head(AUDIT_N)
    _fill_n = AUDIT_N - len(_top)
    _rest = _pool.drop(_top.index)
    _fill = _rest.sample(n=min(_fill_n, len(_rest)), random_state=AUDIT_SEED) if _fill_n > 0 else _rest.iloc[0:0]
    audit_df = pd.concat([_top, _fill]).reset_index(drop=True)
    assert len(audit_df) <= AUDIT_N, f'{len(audit_df):,} audit rows exceeds AUDIT_N ({AUDIT_N:,})'
    print(f'`none`-only rows: {len(_pool):,} of {len(ok):,}; with any engagement: {len(_engaged):,}')
    print(f'Selected {len(audit_df):,} for audit: {len(_top):,} by engagement'
          + (f' + {len(_fill):,} random fill (seed {AUDIT_SEED})' if len(_fill) else '')
          + f'; engagement range {audit_df["_engagement"].min():,.0f}–{audit_df["_engagement"].max():,.0f}')
    _a_in, _a_out = (BATCH_RATE_FACTOR * _r for _r in MODEL_RATES.get(AUDIT_MODEL, FALLBACK_RATES))
    _a_est_in  = sum(len(build_prompt(t)) for t in audit_df['text']) / CHARS_PER_TOKEN
    _a_est_out = len(audit_df) * AUDIT_MAX_OUTPUT_TOKENS
    audit_est_cost = _a_est_in / 1e6 * _a_in + _a_est_out / 1e6 * _a_out
    print(f'{len(audit_df):,} batch requests to {AUDIT_MODEL}: est. input {_a_est_in:,.0f} tokens, '
          f'output ≤ {_a_est_out:,.0f} (cap, thinking included); est. cost ≤ ${audit_est_cost:,.2f} '
          f'@ batch rate ${_a_in}/M in, ${_a_out}/M out')
    if audit_est_cost > COST_ALERT_USD:
        print(f'*** COST ALERT: ${audit_est_cost:,.2f} exceeds COST_ALERT_USD (${COST_ALERT_USD:,.2f}). ***')

## Preflight, Submit and Collect the Audit

One synchronous call with the real prompt, then the audit file, one job, and the wait. On a
resumed run the job on record is collected, not resubmitted.

In [ ]:
%%time
AUDIT_OK = False
audit_out: dict = {}
audit_stopped_early = False
state = load_state()
if not AUDIT_ENABLED or not len(audit_df):
    print('Nothing to audit.')
else:
    # Config for the audit model: NO temperature (deprecated on 3.5+/3.6+, ignored today and a
    # 400 tomorrow), a raised output cap, and the lowest thinking level the SDK lets us name.
    # Probed like the main client cell, because models disagree about what they accept.
    _audit_base = {k: v for k, v in config_kwargs.items() if k != 'temperature'}
    _audit_base['max_output_tokens'] = AUDIT_MAX_OUTPUT_TOKENS
    _audit_kwargs = None
    if 'thinking_level' in getattr(types.ThinkingConfig, 'model_fields', {}):
        _try = {**_audit_base, 'thinking_config': types.ThinkingConfig(thinking_level=AUDIT_THINKING_LEVEL)}
        _err = _probe(_try, AUDIT_MODEL)
        if not _err:
            _audit_kwargs = _try
            print(f'{AUDIT_MODEL}: thinking_level={AUDIT_THINKING_LEVEL!r} accepted')
        else:
            print(f'WARNING: {AUDIT_MODEL} rejected thinking_level={AUDIT_THINKING_LEVEL!r} — {_err}')
    else:
        print('This google-genai release has no ThinkingConfig.thinking_level — upgrade it to bound '
              'the audit model\'s thinking; running at the model default.')
    if _audit_kwargs is None:
        _err = _probe(_audit_base, AUDIT_MODEL)
        assert not _err, f'{AUDIT_MODEL} rejected the base config too: {_err}'
        _audit_kwargs = _audit_base
        print(f'{AUDIT_MODEL}: thinking at the model default (UNBOUNDED; watch the token report)')
    _audit_config = types.GenerateContentConfig(**_audit_kwargs)

    if state['jobs'].get('audit'):
        # Resuming: the audit file was built and submitted earlier from this same selection.
        assert state['jobs']['audit']['ids'] == list(audit_df['id']), (
            'The audit selection reproduced now differs from the one submitted — AUDIT_N, '
            'AUDIT_SEED or the labels changed. Do not collect that job into this dataframe.')
        print(f'Resuming audit job {state["jobs"]["audit"]["name"]}')
    else:
        # Synchronous preflight with the real prompt — one call, before the job is billed.
        _r = client.models.generate_content(model=AUDIT_MODEL, contents=build_prompt(_probe_text),
                                            config=_audit_config)
        _parsed = json.loads(_strip_code_fences(_r.text or ''))
        for p in _parsed:
            assert p['category'] in CATEGORIES, f'category {p["category"]!r} not in CATEGORIES'
        _m = getattr(_r, 'usage_metadata', None)
        print(f'Preflight OK: {_parsed}')
        if _m is not None:
            print(f'  tokens: input={_m.prompt_token_count}, output={_m.candidates_token_count}, '
                  f'thinking={getattr(_m, "thoughts_token_count", None)}')
        # Both audit ceilings, pre-flight: the row cap as the line count of the file, the
        # token backstop against the upper-bound estimate. Neither can act after upload.
        assert len(audit_df) <= AUDIT_N, f'{len(audit_df):,} audit rows exceeds AUDIT_N ({AUDIT_N:,})'
        if _a_est_in + _a_est_out > AUDIT_MAX_TOKENS:
            raise RuntimeError(
                f'Audit upper-bound estimate {_a_est_in + _a_est_out:,.0f} tokens exceeds '
                f'AUDIT_MAX_TOKENS ({AUDIT_MAX_TOKENS:,}). Nothing uploaded.')
        _cfg  = gen_config_json(_audit_kwargs)
        _path = BATCH_FOLDER / f'llm_bootstrap_batch_{RUN_STAMP}_audit.jsonl'
        write_jsonl(_path, [build_request(row['id'], row['text'], _cfg) for _, row in audit_df.iterrows()])
        _n = count_lines(_path)
        assert _n == len(audit_df) and _n <= AUDIT_N, f'{_path.name}: {_n:,} lines (cap {AUDIT_N:,})'
        _job, _file = submit_file_job(AUDIT_MODEL, _path, f'llm_bootstrap_{RUN_STAMP}_audit')
        state['jobs']['audit'] = {'name': _job.name, 'model': AUDIT_MODEL, 'input_file': _file,
                                  'requests': _n, 'ids': list(audit_df['id']),
                                  'submitted': time.strftime('%Y-%m-%dT%H:%M:%S')}
        save_state(state)          # before anything else can fail
        print(f'audit: {describe_job(_job)}')
    AUDIT_OK = True

    _audit_usage_start = dict(_usage)
    _t0 = time.time()
    _an = state['jobs']['audit']['name']
    _j  = wait_for_jobs([_an], BATCH_WAIT_MAX_S, BATCH_POLL_S)[_an]
    print(describe_job(_j))
    _st = _state_name(_j)
    if _st not in TERMINAL_STATES:
        raise RuntimeError(
            f'Audit job still running after {BATCH_WAIT_MAX_S / 60:.0f} min. Nothing is lost: set '
            f'RESUME_RUN_STAMP = {RUN_STAMP!r} in Configuration and re-run from the top later.')
    if _st in FAILED_STATES:
        audit_stopped_early = True
        print(f'*** audit job ended {_st}: {getattr(_j, "error", None)} — no audit rows. ***')
    else:
        _rows = fetch_results(_j)
        _out = BATCH_FOLDER / f'llm_bootstrap_batch_{RUN_STAMP}_audit_results.jsonl'
        write_jsonl(_out, _rows)
        audit_out, _errs = parse_results(_rows, 'audit')
        state['jobs']['audit'].update({'state': _st, 'result_file': _out.name, 'rows': len(_rows),
                                       'failed_rows': len(_errs),
                                       'collected': time.strftime('%Y-%m-%dT%H:%M:%S')})
        save_state(state)
    # Audited rows are seen rows too — the uploaded file is the record, parsed or not.
    sent_ids = sorted(set(globals().get('sent_ids', [])) | set(audit_df['id']))

    # Merge beside the operative columns. Non-audited rows stay blank, so a reviewer can tell
    # "not audited" from "audited and agreed".
    out_df['audited'] = out_df['id'].isin(audit_out).astype(int)
    for cat in CATEGORIES:
        out_df[f'audit_{cat}'] = np.nan
    out_df['audit_confidence'] = np.nan
    out_df['audit_rationale']  = ''
    out_df['audit_model']      = ''
    _ix = out_df['id'].map(lambda i: i in audit_out)
    for idx in out_df.index[_ix]:
        _preds = audit_out[out_df.at[idx, 'id']]
        _cats  = {x['category'] for x in _preds}
        for cat in CATEGORIES:
            out_df.at[idx, f'audit_{cat}'] = 1.0 if cat in _cats else 0.0
        out_df.at[idx, 'audit_confidence'] = max([float(x.get('confidence', 0.0)) for x in _preds] or [0.0])
        out_df.at[idx, 'audit_rationale']  = ' | '.join(str(x.get('rationale', '')) for x in _preds)
        out_df.at[idx, 'audit_model']      = AUDIT_MODEL
    _a_usage = {k: _usage[k] - _audit_usage_start[k] for k in _usage}
    _a_in, _a_out = (BATCH_RATE_FACTOR * _r for _r in MODEL_RATES.get(AUDIT_MODEL, FALLBACK_RATES))
    audit_usage_record = {
        'model': AUDIT_MODEL, 'thinking_level': AUDIT_THINKING_LEVEL if 'thinking_config' in _audit_kwargs else None,
        'max_output_tokens': AUDIT_MAX_OUTPUT_TOKENS, 'rows_selected': int(len(audit_df)),
        'rows_audited': int(len(audit_out)), 'stopped_early': audit_stopped_early,
        'selection': 'none-only rows by likes+retweets desc, seeded random fill',
        'batch_job': _an, 'batch_rate_factor': BATCH_RATE_FACTOR,
        'usage': _a_usage,
        'cost_usd': round(_a_usage['input'] / 1e6 * _a_in + _a_usage['output'] / 1e6 * _a_out, 4),
    }
    print(f"\n{'FAILED' if audit_stopped_early else 'Done'}: {len(audit_out):,} rows audited in "
          f"{time.time() - _t0:.0f}s; {_a_usage['calls']:,} result rows, {_a_usage['input']:,} in / "
          f"{_a_usage['output']:,} out tokens (thinking included) = ${audit_usage_record['cost_usd']:,.4f}"
          f" (est. ≤ ${audit_est_cost:,.2f})")

## Audit Report

In [ ]:
%%time
# Nothing here calls the API; re-run freely. Reads out_df's audit_* columns.
N_AUDIT_SAMPLES = 15
if 'audited' not in out_df or not int(out_df['audited'].sum()):
    print('No audited rows.')
else:
    _au = out_df[out_df['audited'] == 1].copy()
    _subst = [c for c in CATEGORIES if c != 'none']
    _au['_relabelled'] = _au[[f'audit_{c}' for c in _subst]].sum(axis=1) > 0
    _au['_parse_err']  = _au[[f'audit_{c}' for c in CATEGORIES]].sum(axis=1) == 0
    print('AUDIT REPORT — second opinion on `none`')
    print('-' * 80)
    print(f"  audited {len(_au):,} `none`-only rows with {_au['audit_model'].iloc[0]}")
    print(f"  still `none`     : {int((~_au['_relabelled'] & ~_au['_parse_err']).sum()):>5,}")
    print(f"  given a label    : {int(_au['_relabelled'].sum()):>5,}  "
          f"({_au['_relabelled'].mean():.1%} — the false-negative rate of pass 1 on these rows, "
          f"as judged by the audit model)")
    print(f"  parse errors     : {int(_au['_parse_err'].sum()):>5,}")
    print('\n  Labels the audit model assigned:')
    for cat in _subst:
        n = int(_au[f'audit_{cat}'].sum())
        if n:
            print(f'    {cat:<20} {n:>5,}  mean audit confidence '
                  f'{_au.loc[_au[f"audit_{cat}"] == 1, "audit_confidence"].mean():.2f}')
    _eng = pd.to_numeric(_au['likes'], errors='coerce').fillna(0) + pd.to_numeric(_au['retweets'], errors='coerce').fillna(0)
    for name, m in (('relabelled', _au['_relabelled']), ('kept none', ~_au['_relabelled'] & ~_au['_parse_err'])):
        if m.any():
            print(f'  median engagement, {name:<10}: {_eng[m].median():,.0f}')
    print(f'\n  EXAMPLES — {min(N_AUDIT_SAMPLES, int(_au["_relabelled"].sum()))} relabelled rows, highest audit confidence first')
    print('-' * 80)
    for _, r in _au[_au['_relabelled']].sort_values('audit_confidence', ascending=False).head(N_AUDIT_SAMPLES).iterrows():
        _labs = [c for c in CATEGORIES if r[f'audit_{c}'] == 1]
        print(f"  audit: [{', '.join(_labs)}]  conf={r['audit_confidence']:.2f}  likes={r.get('likes', 0)}  rt={r.get('retweets', 0)}")
        for line in textwrap.wrap(str(r['text']), 80 - 6) or ['']:
            print(f'      {line}')
        for line in textwrap.wrap(f"pass 1 said none: {r.get('rationale', '')}", 80 - 6):
            print(f'      | {line}')
        for line in textwrap.wrap(f"audit: {r['audit_rationale']}", 80 - 6):
            print(f'      | {line}')
        print()

# 7 · Save

Four artifacts, named exactly as `01` names them so `02` and `03` pick them up: the
HITL-schema CSV, the full pickle with `confidence` and `rationale`, two JSON records —
measured token usage (with the job names, so a figure can be traced to its batch) and the
accumulating basket of tweet ids the LLM has seen — and a verbatim copy of the prompt,
byte-identical to the repo's. The request files, result files and state file are already
on Drive under `batch/`.
The CSV is ordered positive-class-first, and carries the model's `rationale` so a
reviewer can see why each label was chosen.
**The basket is training data: exclude those ids from any held-out evaluation.**

## Outputs, Usage Record, and Seen-Ids Basket

In [ ]:
%%time
import hashlib

_label_rank = {_lab: _i for _i, _lab in enumerate(CATEGORIES)}
out_df = out_df.sort_values(
    'confidence',
    ascending=False,
    kind='stable',
).reset_index(drop=True)

# Flatten rationale
for _rcol in [c for c in out_df.columns if c == 'rationale' or c.startswith('rationale_pass')]:
    out_df[_rcol] = (out_df[_rcol].astype(str)
                     .str.replace(r'\s+', ' ', regex=True).str.strip())

_disagree_cols = []
if 'passes_agree' in out_df.columns:
    _disagree_cols = ['passes_agree']

_pass_models = list(PASS_MODELS_RESOLVED[:len(passes)]) if 'passes' in globals() else [MODEL_NAME]
_provenance_cols = []
for _p in range(len(_pass_models)):
    out_df[f'model_pass{_p + 1}'] = _pass_models[_p]
    out_df[f'temp_pass{_p + 1}'] = _temps[_p] if '_temps' in globals() else TEMPERATURE
    _provenance_cols += [f'model_pass{_p + 1}', f'temp_pass{_p + 1}']

human_cols = [f'human_{c}' for c in CATEGORIES]
pred_cols = [f'pred_{c}' for c in CATEGORIES]

# Audit columns (section 6) ride along when present; 02/03 select by name and ignore them.
_audit_cols = ([c for c in ['audited'] + [f'audit_{c}' for c in CATEGORIES]
                + ['audit_confidence', 'audit_rationale', 'audit_model'] if c in out_df.columns])
hitl_schema_cols = (['id', 'text', 'likes', 'retweets'] + pred_cols + ['rationale']
                    + _disagree_cols + human_cols + _audit_cols + _provenance_cols)
hitl_schema_cols = [c for c in hitl_schema_cols if c in out_df.columns]

out_df[hitl_schema_cols].to_csv(OUTPUT_CSV, index=False)
out_df.to_pickle(OUTPUT_PKL)

n_errors = int(out_df['parse_error'].sum())
print(f'Saved → {OUTPUT_CSV}')
print(f'Saved → {OUTPUT_PKL}')
print(f'PARSE_ERROR rows: {n_errors:,} / {len(out_df):,} ({n_errors/max(len(out_df),1):.1%})')

usage_record = {
    'timestamp':       time.strftime('%Y-%m-%dT%H:%M:%S'),
    'run_stamp':       RUN_STAMP,
    'labels_csv':      OUTPUT_CSV.name,
    'model':           MODEL_NAME,
    'dataset_type':    DATASET_TYPE,
    'rows':            int(len(out_df)),
    'prompt_chars':    len(PROMPT_TEMPLATE),
    'prompt_doc_sha256':  PROMPT_DOC_SHA256,
    'prompt_sent_sha256': hashlib.sha256(PROMPT_TEMPLATE.encode('utf-8')).hexdigest()[:12],
    'parse_errors':    int(n_errors),
    'mode':            'batch',
    'batch_rate_factor': BATCH_RATE_FACTOR,
    'batch_state':     BATCH_STATE_JSON.name,
    'batch_jobs':      {k: v['name'] for k, v in load_state().get('jobs', {}).items()},
    'pass_models':     list(PASS_MODELS_RESOLVED[:len(passes)]) if 'passes' in globals() else [MODEL_NAME],
    'pass_temperatures': list(_temps[:len(passes)]) if '_temps' in globals() else [TEMPERATURE],
    'stopped_early':   bool(globals().get('stopped_early', False)),
    # Measured over every result row parsed this session: the passes plus the audit
    # (whose own share is under 'audit'). The cost is at the passes' batch rate.
    'usage':           dict(_usage),
    'cost_usd':        round(usage_cost(), 4),
    'audit':           globals().get('audit_usage_record'),   # None when the audit did not run
}
USAGE_JSON = hitl_folder / f'llm_bootstrap_usage_{RUN_STAMP}.json'
with open(USAGE_JSON, 'w') as f:
    json.dump(usage_record, f, indent=2)
print(f'Saved → {USAGE_JSON}')

SEEN_IDS_JSON = hitl_folder / f'llm_bootstrap_seen_ids_{DATASET_TYPE}.json'
seen = {
    'note': 'Tweet ids sent to the LLM for bootstrap labelling (01 and 01b). Accumulates across runs.',
    'dataset_type': DATASET_TYPE,
    'ids': [],
    'parse_error_ids': [],
    'runs': [],
}
if SEEN_IDS_JSON.exists():
    with open(SEEN_IDS_JSON) as f:
        seen = json.load(f)

run_ids = [str(i) for i in globals().get('sent_ids', out_df['id'])]
err_ids = {str(i) for i in out_df.loc[out_df['parse_error'] == 1, 'id']}
ok_ids  = {str(i) for i in out_df.loc[out_df['parse_error'] == 0, 'id']}

seen['ids'] = sorted(set(seen['ids']) | set(run_ids))
_old_errs = set(seen['parse_error_ids'])
seen['parse_error_ids'] = sorted((_old_errs | err_ids) - ok_ids)
seen['runs'].append(usage_record)

with open(SEEN_IDS_JSON, 'w') as f:
    json.dump(seen, f, indent=2)
print(f'Basket updated → {SEEN_IDS_JSON} (total {len(seen["ids"]):,} seen)')

## Prompt Record for Reviewers

Writes `llm_bootstrap_prompt.md` next to the CSV — the same file that lives in the repo, byte
for byte — so whoever fills in `human_label` is reading exactly the text the model was given.

Nothing run-specific is added to it. A generated header would make the Drive copy differ from
the repo copy, and then neither could be trusted as *the* prompt. The run's model, passes,
temperatures and token spend go into the usage record instead, keyed by the same `sha256`.

In [ ]:
%%time
# PROMPT_DOC is llm_bootstrap_prompt.md, embedded verbatim by sync_prompt.py. Writing it
# out unchanged is what makes the reviewer's copy and the author's copy the same artifact:
# a reviewer reading it in Drive and an author editing it in git see identical bytes, and
# the assertion below is what turns that from an intention into a checked property.
PROMPT_MD_OUT = hitl_folder / 'llm_bootstrap_prompt.md'
PROMPT_MD_OUT.write_text(PROMPT_DOC, encoding='utf-8')
assert PROMPT_MD_OUT.read_text(encoding='utf-8') == PROMPT_DOC, 'prompt record did not round-trip'

print(f'Saved → {PROMPT_MD_OUT}')
print(f'  {len(PROMPT_DOC):,} chars, sha256:{PROMPT_DOC_SHA256} — byte-identical to '
      f'the repo\'s notebooks/05_Classifiers/llm_bootstrap_prompt.md')

# 8 · Teardown

Releases the Colab runtime when `AUTO_DISCONNECT` is True. A submitted job keeps running
after the runtime is gone — that is the point — and is collected with `RESUME_RUN_STAMP`. Keep it **False** while iterating —
unassigning destroys every variable, so a failed run leaves nothing to inspect.

## Disconnect

In [ ]:
# Disconnect from Colab runtime (no-op locally).
# Gated by AUTO_DISCONNECT: unassigning frees the runtime but DESTROYS all variables,
# so a failed run leaves nothing to inspect. Keep it False while iterating; set it True
# for a long unattended full run where releasing the instance matters.
if AUTO_DISCONNECT:
    try:
        from google.colab import runtime
        runtime.unassign()
    except ImportError:
        pass
else:
    print('AUTO_DISCONNECT is False — runtime left running so out_df stays inspectable.')
